# Forecasting NSW electricity prices from five-minute market data

## Public reproducibility notebook

This notebook reproduces the frozen Assignment 2 experiment: a daily seasonal-naive
baseline, multi-output Ridge regression and multi-output Random Forest for the next 96
five-minute NSW1 prices from a 288-step history. It downloads the fixed `a2-data-v1`
release, verifies the exact byte size and SHA-256 checksum **before** pandas loads it,
then enforces continuity, split-containment and training-only preprocessing controls.

The notebook is an executable record, not a new model-development environment. All
features, candidates, selection rules, thresholds, interpretations and expected
aggregate results were frozen before the held-out test period was evaluated. A
reproduction mismatch stops execution and never updates those frozen values.

**Resource note.** The complete Random Forest sequence needs at least 4 GiB and 30%
available system memory; 5 GiB is recommended. Isolated workers enforce the approved
memory and runtime limits. The safeguards are not relaxed after a failure.

Data source: Australian Energy Market Operator (AEMO), NEMWeb market data. This
dataset is an assignment-specific prepared NSW1 extract and is not an official AEMO
redistribution format. AEMO material is used with attribution in accordance with
AEMO's Copyright Permissions.


## 1. Environment

Install the exact package versions used for the frozen run. Run this cell before any
imports. If the notebook environment requests a runtime restart, restart and run all
cells again from the top.


In [ ]:
%pip install -q numpy==1.26.4 pandas==2.3.3 scipy==1.15.3 scikit-learn==1.7.2 joblib==1.5.3 psutil==7.1.3


In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib
import json
import platform
import shutil
import sys
import tempfile
import urllib.request
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import psutil
import scipy
import sklearn

EXPECTED_VERSIONS = {
    "NumPy": "1.26.4", "pandas": "2.3.3", "SciPy": "1.15.3",
    "scikit-learn": "1.7.2", "joblib": "1.5.3", "psutil": "7.1.3",
}
observed_versions = {
    "NumPy": np.__version__, "pandas": pd.__version__, "SciPy": scipy.__version__,
    "scikit-learn": sklearn.__version__, "joblib": joblib.__version__,
    "psutil": psutil.__version__,
}
if observed_versions != EXPECTED_VERSIONS:
    raise RuntimeError(f"Package-version mismatch: {observed_versions} != {EXPECTED_VERSIONS}")

RUNTIME_ROOT = Path(tempfile.gettempdir()) / "nsw_electricity_a2_public_runtime"
if RUNTIME_ROOT.exists():
    shutil.rmtree(RUNTIME_ROOT)
CODE_DIR = RUNTIME_ROOT / "code"
DATA_DIR = RUNTIME_ROOT / "data" / "processed"
OUTPUT_DIR = RUNTIME_ROOT / "outputs"
for path in (CODE_DIR, DATA_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

print({"Python": platform.python_version(), **observed_versions})
print(f"Transient runtime: {RUNTIME_ROOT}")


## 2. Validated implementation modules

The following cell materialises the authoritative Phase 5–7 code into the transient
runtime. The source hashes are recorded so that the public notebook can be compared
with the frozen local implementation.


In [ ]:
MODULE_SOURCES = {'feature_engineering.py': '"""Leakage-safe feature construction for the NSW electricity-price A2.\n\nThe module builds approved period matrices in memory. It never persists row-level\nsamples and deliberately contains no scaler, model, prediction, or metric code. All\ntemporal indexing is relative to the forecast origin at ``t``.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom hashlib import sha256\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport numpy as np\nimport pandas as pd\n\n\nAPPROVED_SHA256 = "d9a3059fb517eabbdabfa2016b1195fef29eeff0bce00c0b60d7defeab0cddb7"\nPUBLIC_DATA_URL = (\n    "https://github.com/Yai-Git/nsw-electricity-price-data-a2/releases/download/"\n    "a2-data-v1/prices_NSW1_5min_prepared.csv"\n)\nEXPECTED_COLUMNS = [\n    "date_time",\n    "RRP",\n    "RUNNO",\n    "PERIODID",\n    "PRICE_STATUS",\n    "INVALIDFLAG",\n    "continuity_segment_id",\n]\n\nHISTORY_STEPS = 288\nTARGET_STEPS = 96\nRECENT_LAGS = np.array([0, 1, 2, 3, 6, 12, 24, 48, 72, 96, 144, 180])\nROLLING_WINDOWS = (12, 36, 72, 144, 288)\nROLLING_STATS = ("mean", "std", "min", "max")\nHORIZONS = np.arange(1, TARGET_STEPS + 1)\n\nRECENT_FEATURE_NAMES = [f"rrp_lag_{lag:03d}" for lag in RECENT_LAGS]\nCALENDAR_FEATURE_NAMES = [\n    "origin_slot_sin",\n    "origin_slot_cos",\n    "origin_dow_sin",\n    "origin_dow_cos",\n    "origin_doy_sin",\n    "origin_doy_cos",\n]\nROLLING_FEATURE_NAMES = [\n    f"rrp_roll_{stat}_{window:03d}"\n    for window in ROLLING_WINDOWS\n    for stat in ROLLING_STATS\n]\nPREVIOUS_DAY_FEATURE_NAMES = [f"prev_day_rrp_h{h:03d}" for h in HORIZONS]\nFEATURE_NAMES = (\n    RECENT_FEATURE_NAMES\n    + CALENDAR_FEATURE_NAMES\n    + ROLLING_FEATURE_NAMES\n    + PREVIOUS_DAY_FEATURE_NAMES\n)\nTARGET_NAMES = [f"target_rrp_h{h:03d}" for h in HORIZONS]\nFEATURE_SET_SLICES = {"F1": slice(0, 18), "F2": slice(0, 38), "F3": slice(0, 134)}\n\nSPLITS = {\n    "training": (pd.Timestamp("2021-10-01 00:05:00"), pd.Timestamp("2023-12-31 23:55:00")),\n    "validation": (pd.Timestamp("2024-01-01 00:00:00"), pd.Timestamp("2024-06-30 23:55:00")),\n    "development": (pd.Timestamp("2021-10-01 00:05:00"), pd.Timestamp("2024-06-30 23:55:00")),\n    "test": (pd.Timestamp("2024-07-01 00:00:00"), pd.Timestamp("2025-01-22 15:50:00")),\n}\nEXPECTED_SAMPLE_COUNTS = {\n    "training": 235_575,\n    "validation": 52_033,\n    "development": 287_991,\n    "test": 58_848,\n}\n\nPRICE_TOLERANCE = 0.002\nCALENDAR_TOLERANCE = 1e-7\nRANDOM_SEED = 42\n\n\n@dataclass\nclass MatrixBundle:\n    """In-memory matrices and minimal metadata for one approved split."""\n\n    split: str\n    X: np.ndarray\n    Y: np.ndarray\n    origin_timestamps: np.ndarray\n    origin_positions: np.ndarray\n    segment_ids: np.ndarray\n    block_summary: pd.DataFrame\n    conversion_summary: pd.DataFrame\n\n    @property\n    def F1(self) -> np.ndarray:\n        return self.X[:, FEATURE_SET_SLICES["F1"]]\n\n    @property\n    def F2(self) -> np.ndarray:\n        return self.X[:, FEATURE_SET_SLICES["F2"]]\n\n    @property\n    def F3(self) -> np.ndarray:\n        return self.X[:, FEATURE_SET_SLICES["F3"]]\n\n\ndef sha256_file(path: Path, chunk_bytes: int = 1 << 20) -> str:\n    digest = sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(chunk_bytes), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef resolve_dataset(local_path: Path, fallback_path: Path | None = None) -> tuple[Path, str]:\n    """Return a checksum-verified local file, downloading only from the fixed URL if needed."""\n\n    if local_path.exists():\n        digest = sha256_file(local_path)\n        if digest != APPROVED_SHA256:\n            raise RuntimeError(\n                f"Canonical dataset checksum mismatch at {local_path}: {digest}; "\n                f"expected {APPROVED_SHA256}."\n            )\n        return local_path, "local canonical file"\n\n    if fallback_path is None:\n        raise FileNotFoundError(\n            f"Local canonical dataset not found at {local_path}, and no fallback path was supplied."\n        )\n\n    import requests\n\n    fallback_path.parent.mkdir(parents=True, exist_ok=True)\n    response = requests.get(PUBLIC_DATA_URL, timeout=120)\n    response.raise_for_status()\n    digest = sha256(response.content).hexdigest()\n    if digest != APPROVED_SHA256:\n        raise RuntimeError(\n            f"Downloaded dataset checksum mismatch: {digest}; expected {APPROVED_SHA256}. "\n            "The file was not loaded."\n        )\n    fallback_path.write_bytes(response.content)\n    return fallback_path, "fixed GitHub Release asset"\n\n\ndef load_canonical_dataset(path: Path) -> pd.DataFrame:\n    """Load the already checksum-verified seven-column canonical dataset."""\n\n    frame = pd.read_csv(path, parse_dates=["date_time"])\n    if frame.columns.tolist() != EXPECTED_COLUMNS:\n        raise ValueError(f"Unexpected schema: {frame.columns.tolist()}")\n    if frame["date_time"].isna().any():\n        raise ValueError("Timestamp parsing failed for at least one row.")\n    if frame["date_time"].dt.tz is not None:\n        raise ValueError("date_time must remain timezone-naive under the fixed NEM/AEST convention.")\n    if frame.isna().any().any():\n        raise ValueError("The canonical dataset contains missing field values.")\n    if not frame["date_time"].is_monotonic_increasing:\n        raise ValueError("Canonical timestamps are not chronological.")\n    return frame\n\n\ndef calendar_features(timestamps: Iterable[pd.Timestamp] | pd.DatetimeIndex) -> np.ndarray:\n    """Return six float64 cyclic features under the fixed NEM/AEST convention."""\n\n    idx = pd.DatetimeIndex(timestamps)\n    if idx.tz is not None:\n        raise ValueError("Calendar timestamps must be timezone-naive.")\n    if np.any(idx.second != 0) or np.any(idx.minute % 5 != 0):\n        raise ValueError("Forecast origins must lie exactly on five-minute boundaries.")\n\n    slot = 12 * idx.hour.to_numpy(dtype=np.int64) + idx.minute.to_numpy(dtype=np.int64) // 5\n    dow = idx.dayofweek.to_numpy(dtype=np.int64)  # Monday = 0.\n    doy0 = idx.dayofyear.to_numpy(dtype=np.int64) - 1\n    years = idx.year.to_numpy(dtype=np.int64)\n    leap = (years % 4 == 0) & ((years % 100 != 0) | (years % 400 == 0))\n    days_in_year = np.where(leap, 366.0, 365.0)\n\n    return np.column_stack(\n        [\n            np.sin(2.0 * np.pi * slot / 288.0),\n            np.cos(2.0 * np.pi * slot / 288.0),\n            np.sin(2.0 * np.pi * dow / 7.0),\n            np.cos(2.0 * np.pi * dow / 7.0),\n            np.sin(2.0 * np.pi * doy0 / days_in_year),\n            np.cos(2.0 * np.pi * doy0 / days_in_year),\n        ]\n    ).astype(np.float64, copy=False)\n\n\ndef feature_catalog() -> pd.DataFrame:\n    """Describe all 134 predictors and their forecast-time source offsets."""\n\n    rows: list[dict] = []\n    for lag, name in zip(RECENT_LAGS, RECENT_FEATURE_NAMES):\n        rows.append(\n            {\n                "feature_index": len(rows),\n                "feature_name": name,\n                "feature_group": "recent_lag",\n                "source_min_offset": -int(lag),\n                "source_max_offset": -int(lag),\n                "formula": f"RRP_(t-{int(lag)})" if lag else "RRP_t",\n            }\n        )\n    for name, formula in zip(\n        CALENDAR_FEATURE_NAMES,\n        [\n            "sin(2*pi*slot/288)",\n            "cos(2*pi*slot/288)",\n            "sin(2*pi*dow/7), Monday=0",\n            "cos(2*pi*dow/7), Monday=0",\n            "sin(2*pi*(dayofyear-1)/days_in_year)",\n            "cos(2*pi*(dayofyear-1)/days_in_year)",\n        ],\n    ):\n        rows.append(\n            {\n                "feature_index": len(rows),\n                "feature_name": name,\n                "feature_group": "origin_calendar",\n                "source_min_offset": 0,\n                "source_max_offset": 0,\n                "formula": formula,\n            }\n        )\n    for window in ROLLING_WINDOWS:\n        for stat in ROLLING_STATS:\n            rows.append(\n                {\n                    "feature_index": len(rows),\n                    "feature_name": f"rrp_roll_{stat}_{window:03d}",\n                    "feature_group": "rolling",\n                    "source_min_offset": -(window - 1),\n                    "source_max_offset": 0,\n                    "formula": (\n                        f"{stat}(RRP_(t-{window - 1}):RRP_t, ddof=0)"\n                        if stat == "std"\n                        else f"{stat}(RRP_(t-{window - 1}):RRP_t)"\n                    ),\n                }\n            )\n    for horizon, name in zip(HORIZONS, PREVIOUS_DAY_FEATURE_NAMES):\n        offset = int(horizon - HISTORY_STEPS)\n        rows.append(\n            {\n                "feature_index": len(rows),\n                "feature_name": name,\n                "feature_group": "previous_day",\n                "source_min_offset": offset,\n                "source_max_offset": offset,\n                "formula": f"RRP_(t{offset:+d})",\n            }\n        )\n    catalog = pd.DataFrame(rows)\n    catalog["forecast_time_available"] = catalog["source_max_offset"] <= 0\n    return catalog\n\n\ndef target_catalog() -> pd.DataFrame:\n    return pd.DataFrame(\n        {\n            "target_index": np.arange(TARGET_STEPS),\n            "target_name": TARGET_NAMES,\n            "horizon": HORIZONS,\n            "minutes_after_origin": HORIZONS * 5,\n            "source_offset": HORIZONS,\n            "formula": [f"RRP_(t+{h})" for h in HORIZONS],\n        }\n    )\n\n\ndef feature_set_catalog() -> pd.DataFrame:\n    return pd.DataFrame(\n        [\n            {\n                "feature_set": "F1",\n                "column_start": 0,\n                "column_end_inclusive": 17,\n                "feature_count": 18,\n                "groups": "recent lags + origin calendar",\n                "purpose": "predeclared Ridge ablation",\n            },\n            {\n                "feature_set": "F2",\n                "column_start": 0,\n                "column_end_inclusive": 37,\n                "feature_count": 38,\n                "groups": "F1 + rolling statistics",\n                "purpose": "predeclared Ridge ablation",\n            },\n            {\n                "feature_set": "F3",\n                "column_start": 0,\n                "column_end_inclusive": 133,\n                "feature_count": 134,\n                "groups": "F2 + previous-day prices",\n                "purpose": "fixed final Ridge-versus-Random-Forest comparison",\n            },\n        ]\n    )\n\n\ndef _cast_difference(values64: np.ndarray) -> tuple[np.ndarray, float]:\n    values32 = values64.astype(np.float32)\n    maximum = float(np.max(np.abs(values64 - values32.astype(np.float64)))) if values64.size else 0.0\n    return values32, maximum\n\n\ndef _split_mask(frame: pd.DataFrame, split: str) -> np.ndarray:\n    if split not in SPLITS:\n        raise ValueError(f"Only approved periods are accepted, received {split!r}.")\n    start, end = SPLITS[split]\n    return ((frame["date_time"] >= start) & (frame["date_time"] <= end)).to_numpy()\n\n\ndef build_split_matrices(frame: pd.DataFrame, split: str) -> MatrixBundle:\n    """Build one approved split blockwise without crossing gaps or split boundaries."""\n\n    mask = _split_mask(frame, split)\n    positions = np.flatnonzero(mask)\n    split_frame = frame.loc[mask].copy()\n    split_frame["_global_position"] = positions\n\n    block_specs = []\n    for segment_id, block in split_frame.groupby("continuity_segment_id", sort=False):\n        times = block["date_time"].to_numpy()\n        if len(times) > 1:\n            deltas = np.diff(times).astype("timedelta64[m]").astype(np.int64)\n            if not np.all(deltas == 5):\n                raise ValueError(f"Non-five-minute spacing in {split} segment {segment_id}.")\n        count = max(0, len(block) - (HISTORY_STEPS + TARGET_STEPS) + 1)\n        block_specs.append((int(segment_id), block, count))\n\n    sample_count = sum(item[2] for item in block_specs)\n    expected_count = EXPECTED_SAMPLE_COUNTS[split]\n    if sample_count != expected_count:\n        raise ValueError(f"{split} sample count {sample_count} != approved {expected_count}.")\n\n    X = np.empty((sample_count, len(FEATURE_NAMES)), dtype=np.float32)\n    Y = np.empty((sample_count, TARGET_STEPS), dtype=np.float32)\n    origin_timestamps = np.empty(sample_count, dtype="datetime64[ns]")\n    origin_positions = np.empty(sample_count, dtype=np.int64)\n    segment_ids = np.empty(sample_count, dtype=np.int16)\n\n    conversion_maxima = {\n        "recent_lag_features": 0.0,\n        "origin_calendar_features": 0.0,\n        "rolling_features": 0.0,\n        "previous_day_features": 0.0,\n        "targets": 0.0,\n    }\n    block_rows = []\n    cursor = 0\n\n    for segment_id, block, count in block_specs:\n        if count <= 0:\n            continue\n        rrp = block["RRP"].to_numpy(dtype=np.float64, copy=True)\n        timestamps = pd.DatetimeIndex(block["date_time"])\n        global_positions = block["_global_position"].to_numpy(dtype=np.int64)\n        local_origins = np.arange(HISTORY_STEPS - 1, len(block) - TARGET_STEPS, dtype=np.int64)\n        destination = slice(cursor, cursor + count)\n\n        recent64 = rrp[local_origins[:, None] - RECENT_LAGS[None, :]]\n        recent32, difference = _cast_difference(recent64)\n        conversion_maxima["recent_lag_features"] = max(\n            conversion_maxima["recent_lag_features"], difference\n        )\n        X[destination, 0:12] = recent32\n        del recent64, recent32\n\n        calendar64 = calendar_features(timestamps[local_origins])\n        calendar32, difference = _cast_difference(calendar64)\n        conversion_maxima["origin_calendar_features"] = max(\n            conversion_maxima["origin_calendar_features"], difference\n        )\n        X[destination, 12:18] = calendar32\n        del calendar64, calendar32\n\n        rolling_column = 18\n        series = pd.Series(rrp, copy=False)\n        rolling_differences = []\n        for window in ROLLING_WINDOWS:\n            rolling = series.rolling(window=window, min_periods=window)\n            rolling64 = np.column_stack(\n                [\n                    rolling.mean().to_numpy()[local_origins],\n                    rolling.std(ddof=0).to_numpy()[local_origins],\n                    rolling.min().to_numpy()[local_origins],\n                    rolling.max().to_numpy()[local_origins],\n                ]\n            )\n            rolling32, difference = _cast_difference(rolling64)\n            rolling_differences.append(difference)\n            X[destination, rolling_column : rolling_column + 4] = rolling32\n            rolling_column += 4\n            del rolling64, rolling32, rolling\n        conversion_maxima["rolling_features"] = max(\n            conversion_maxima["rolling_features"], max(rolling_differences)\n        )\n\n        previous64 = rrp[local_origins[:, None] + HORIZONS[None, :] - HISTORY_STEPS]\n        previous32, difference = _cast_difference(previous64)\n        conversion_maxima["previous_day_features"] = max(\n            conversion_maxima["previous_day_features"], difference\n        )\n        X[destination, 38:134] = previous32\n        del previous64, previous32\n\n        target64 = rrp[local_origins[:, None] + HORIZONS[None, :]]\n        target32, difference = _cast_difference(target64)\n        conversion_maxima["targets"] = max(conversion_maxima["targets"], difference)\n        Y[destination] = target32\n        del target64, target32\n\n        origin_timestamps[destination] = timestamps[local_origins].to_numpy()\n        origin_positions[destination] = global_positions[local_origins]\n        segment_ids[destination] = segment_id\n        block_rows.append(\n            {\n                "split": split,\n                "continuity_segment_id": segment_id,\n                "block_start": timestamps[0],\n                "block_end": timestamps[-1],\n                "record_count": len(block),\n                "eligible_sample_count": count,\n                "first_origin": timestamps[local_origins[0]],\n                "last_origin": timestamps[local_origins[-1]],\n            }\n        )\n        cursor += count\n\n    if cursor != sample_count:\n        raise RuntimeError(f"Internal sample allocation mismatch: {cursor} != {sample_count}.")\n    if not np.isfinite(X).all() or not np.isfinite(Y).all():\n        raise ValueError(f"{split} matrices contain missing or infinite values.")\n\n    conversion_rows = []\n    for group, difference in conversion_maxima.items():\n        tolerance = CALENDAR_TOLERANCE if group == "origin_calendar_features" else PRICE_TOLERANCE\n        conversion_rows.append(\n            {\n                "split": split,\n                "value_group": group,\n                "reference_dtype": "float64",\n                "stored_dtype": "float32",\n                "maximum_absolute_conversion_difference": difference,\n                "tolerance": tolerance,\n                "status": "PASS" if difference <= tolerance else "FAIL",\n            }\n        )\n\n    return MatrixBundle(\n        split=split,\n        X=X,\n        Y=Y,\n        origin_timestamps=origin_timestamps,\n        origin_positions=origin_positions,\n        segment_ids=segment_ids,\n        block_summary=pd.DataFrame(block_rows),\n        conversion_summary=pd.DataFrame(conversion_rows),\n    )\n\n\ndef _expected_feature_vector(frame: pd.DataFrame, position: int) -> tuple[np.ndarray, dict]:\n    rrp = frame["RRP"].to_numpy(dtype=np.float64, copy=False)\n    recent = rrp[position - RECENT_LAGS]\n    timestamp = pd.Timestamp(frame.iloc[position]["date_time"])\n    slot = 12 * timestamp.hour + timestamp.minute // 5\n    dow = timestamp.dayofweek  # Independent reference: Monday = 0.\n    doy0 = timestamp.dayofyear - 1\n    year = timestamp.year\n    is_leap = year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)\n    days_in_year = 366.0 if is_leap else 365.0\n    calendar = np.array(\n        [\n            np.sin(2.0 * np.pi * slot / 288.0),\n            np.cos(2.0 * np.pi * slot / 288.0),\n            np.sin(2.0 * np.pi * dow / 7.0),\n            np.cos(2.0 * np.pi * dow / 7.0),\n            np.sin(2.0 * np.pi * doy0 / days_in_year),\n            np.cos(2.0 * np.pi * doy0 / days_in_year),\n        ],\n        dtype=np.float64,\n    )\n    rolling_values = []\n    rolling_boundaries = []\n    for window in ROLLING_WINDOWS:\n        values = rrp[position - window + 1 : position + 1]\n        rolling_values.extend([values.mean(), values.std(ddof=0), values.min(), values.max()])\n        rolling_boundaries.append((position - window + 1, position))\n    previous = rrp[position + HORIZONS - HISTORY_STEPS]\n    vector = np.concatenate([recent, calendar, np.asarray(rolling_values), previous])\n    return vector, {"rolling_boundaries": rolling_boundaries, "previous": previous}\n\n\ndef _expected_target(frame: pd.DataFrame, position: int) -> np.ndarray:\n    rrp = frame["RRP"].to_numpy(dtype=np.float64, copy=False)\n    return rrp[position + HORIZONS]\n\n\ndef representative_validation(\n    frame: pd.DataFrame, training: MatrixBundle, validation: MatrixBundle\n) -> pd.DataFrame:\n    """Validate every approved mapping for a compact deterministic representative set."""\n\n    bundles = {"training": training, "validation": validation}\n    selected: list[tuple[str, int, str]] = []\n    used: set[tuple[str, int]] = set()\n\n    def add(split: str, row_index: int, reason: str) -> None:\n        key = (split, int(row_index))\n        if key not in used:\n            selected.append((split, int(row_index), reason))\n            used.add(key)\n\n    for segment_id in np.unique(training.segment_ids):\n        indices = np.flatnonzero(training.segment_ids == segment_id)\n        add("training", int(indices[0]), f"first eligible origin in training segment {segment_id}")\n        add("training", int(indices[-1]), f"last eligible origin in training segment {segment_id}")\n    add("validation", 0, "first eligible validation origin")\n    add("validation", len(validation.X) - 1, "last eligible validation origin")\n\n    extreme_positions = np.flatnonzero((frame["RRP"].to_numpy() <= -1000.0) | (frame["RRP"].to_numpy() >= 1000.0))\n    extreme_added = False\n    for event_position in extreme_positions:\n        row_index = int(np.searchsorted(training.origin_positions, event_position, side="left"))\n        while row_index < len(training.origin_positions):\n            origin_position = int(training.origin_positions[row_index])\n            if origin_position > event_position + HISTORY_STEPS - 1:\n                break\n            if (\n                origin_position >= event_position\n                and frame.iloc[origin_position]["continuity_segment_id"]\n                == frame.iloc[event_position]["continuity_segment_id"]\n                and ("training", row_index) not in used\n            ):\n                add("training", row_index, "history contains a legitimate primary-extreme price")\n                extreme_added = True\n                break\n            row_index += 1\n        if extreme_added:\n            break\n    if not extreme_added:\n        raise AssertionError("No additional eligible training history containing an extreme price was found.")\n\n    all_candidates = [\n        (split, index)\n        for split, bundle in bundles.items()\n        for index in range(len(bundle.X))\n        if (split, index) not in used\n    ]\n    rng = np.random.default_rng(RANDOM_SEED)\n    random_indices = rng.choice(len(all_candidates), size=10, replace=False)\n    for ordinal, candidate_index in enumerate(random_indices, start=1):\n        split, row_index = all_candidates[int(candidate_index)]\n        add(split, row_index, f"deterministic seed-42 sample {ordinal:02d}")\n\n    rows = []\n    for representative_id, (split, row_index, reason) in enumerate(selected, start=1):\n        bundle = bundles[split]\n        position = int(bundle.origin_positions[row_index])\n        expected_x, details = _expected_feature_vector(frame, position)\n        expected_y = _expected_target(frame, position)\n        actual_x = bundle.X[row_index].astype(np.float64)\n        actual_y = bundle.Y[row_index].astype(np.float64)\n        lag_error = float(np.max(np.abs(actual_x[0:12] - expected_x[0:12])))\n        calendar_error = float(np.max(np.abs(actual_x[12:18] - expected_x[12:18])))\n        rolling_error = float(np.max(np.abs(actual_x[18:38] - expected_x[18:38])))\n        previous_error = float(np.max(np.abs(actual_x[38:134] - details["previous"])))\n        target_error = float(np.max(np.abs(actual_y - expected_y)))\n        rolling_boundaries_correct = all(\n            start == position - window + 1 and end == position\n            for (start, end), window in zip(details["rolling_boundaries"], ROLLING_WINDOWS)\n        )\n        source_segment = int(frame.iloc[position]["continuity_segment_id"])\n        source_time = pd.Timestamp(frame.iloc[position]["date_time"])\n        split_start, split_end = SPLITS[split]\n        history = frame["RRP"].iloc[position - HISTORY_STEPS + 1 : position + 1].to_numpy()\n        history_contains_primary_extreme = bool(\n            np.any((history <= -1000.0) | (history >= 1000.0))\n        )\n        rows.append(\n            {\n                "representative_id": representative_id,\n                "selection_reason": reason,\n                "split": split,\n                "continuity_segment_id": int(bundle.segment_ids[row_index]),\n                "forecast_origin": pd.Timestamp(bundle.origin_timestamps[row_index]),\n                "all_12_recent_lags_checked": 12,\n                "all_6_calendar_features_checked": 6,\n                "all_20_rolling_features_checked": 20,\n                "rolling_windows_with_boundaries_checked": len(ROLLING_WINDOWS),\n                "all_96_previous_day_mappings_checked": 96,\n                "all_96_target_mappings_checked": 96,\n                "recent_lag_max_abs_error": lag_error,\n                "calendar_max_abs_error": calendar_error,\n                "rolling_max_abs_error": rolling_error,\n                "previous_day_max_abs_error": previous_error,\n                "target_max_abs_error": target_error,\n                "rolling_boundaries_correct": rolling_boundaries_correct,\n                "origin_timestamp_correct": source_time == pd.Timestamp(bundle.origin_timestamps[row_index]),\n                "segment_correct": source_segment == int(bundle.segment_ids[row_index]),\n                "split_correct": split_start <= source_time <= split_end,\n                "history_contains_primary_extreme": history_contains_primary_extreme,\n                "status": "PASS"\n                if (\n                    lag_error <= PRICE_TOLERANCE\n                    and calendar_error <= CALENDAR_TOLERANCE\n                    and rolling_error <= PRICE_TOLERANCE\n                    and previous_error <= PRICE_TOLERANCE\n                    and target_error <= PRICE_TOLERANCE\n                    and rolling_boundaries_correct\n                    and source_time == pd.Timestamp(bundle.origin_timestamps[row_index])\n                    and source_segment == int(bundle.segment_ids[row_index])\n                    and split_start <= source_time <= split_end\n                )\n                else "FAIL",\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef matrix_summary(*bundles: MatrixBundle) -> pd.DataFrame:\n    rows = []\n    for bundle in bundles:\n        for feature_set, feature_slice in FEATURE_SET_SLICES.items():\n            view = bundle.X[:, feature_slice]\n            rows.append(\n                {\n                    "split": bundle.split,\n                    "matrix": feature_set,\n                    "rows": view.shape[0],\n                    "columns": view.shape[1],\n                    "dtype": str(view.dtype),\n                    "memory_mib": view.nbytes / (1024**2),\n                    "shares_memory_with_F3": bool(np.shares_memory(view, bundle.X)),\n                }\n            )\n        rows.append(\n            {\n                "split": bundle.split,\n                "matrix": "Y",\n                "rows": bundle.Y.shape[0],\n                "columns": bundle.Y.shape[1],\n                "dtype": str(bundle.Y.dtype),\n                "memory_mib": bundle.Y.nbytes / (1024**2),\n                "shares_memory_with_F3": False,\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef training_feature_summary(training: MatrixBundle) -> pd.DataFrame:\n    """Return aggregate training-only feature statistics, never row-level values."""\n\n    X = training.X\n    return pd.DataFrame(\n        {\n            "feature_index": np.arange(len(FEATURE_NAMES)),\n            "feature_name": FEATURE_NAMES,\n            "mean": X.mean(axis=0, dtype=np.float64),\n            "std_ddof0": X.std(axis=0, dtype=np.float64),\n            "minimum": X.min(axis=0),\n            "maximum": X.max(axis=0),\n            "finite_count": np.isfinite(X).sum(axis=0),\n        }\n    )\n', 'phase6_experiments.py': '"""Phase 6 training/validation experiments for the NSW electricity-price A2.\n\nThis module deliberately excludes test and unified-development construction. It\nreturns aggregate evidence only and never persists matrices, predictions, scalers,\nor fitted models.\n"""\n\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport json\nimport math\nimport os\nimport platform\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport warnings\nfrom importlib import metadata\nfrom pathlib import Path\nfrom typing import Any\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport psutil\nimport scipy\nimport sklearn\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import Ridge\nfrom sklearn.preprocessing import StandardScaler\n\nimport feature_engineering as fe\n\n\nRIDGE_ALPHAS = (1.0, 10.0, 100.0, 1_000.0, 10_000.0, 100_000.0)\nRIDGE_SETTINGS = {\n    "fit_intercept": True,\n    "solver": "lsqr",\n    "tol": 1e-4,\n    "max_iter": 5000,\n    "copy_X": True,\n}\nTIE_TOLERANCE = 1e-9\n\nRF_COMMON = {\n    "criterion": "squared_error",\n    "min_samples_split": 2,\n    "max_features": 0.5,\n    "bootstrap": True,\n    "oob_score": False,\n    "warm_start": False,\n    "random_state": 42,\n    "n_jobs": -1,\n    "verbose": 0,\n    "ccp_alpha": 0.0,\n}\nRF_CONFIGS = [\n    {"candidate_id": "RF01", "n_estimators": 40, "max_depth": 10, "min_samples_leaf": 20, "max_samples": 0.25, "purpose": "compact reference forest"},\n    {"candidate_id": "RF02", "n_estimators": 40, "max_depth": 16, "min_samples_leaf": 20, "max_samples": 0.25, "purpose": "test greater depth"},\n    {"candidate_id": "RF03", "n_estimators": 40, "max_depth": 16, "min_samples_leaf": 5, "max_samples": 0.25, "purpose": "test finer terminal regions"},\n    {"candidate_id": "RF04", "n_estimators": 40, "max_depth": 16, "min_samples_leaf": 5, "max_samples": 0.50, "purpose": "test more observations per tree"},\n    {"candidate_id": "RF05", "n_estimators": 100, "max_depth": 16, "min_samples_leaf": 5, "max_samples": 0.50, "purpose": "test a larger ensemble"},\n    {"candidate_id": "RF06", "n_estimators": 100, "max_depth": 20, "min_samples_leaf": 5, "max_samples": 0.50, "purpose": "highest approved capacity"},\n]\n\nGIB = 1024**3\nRF_PREFLIGHT_MIN_AVAILABLE_BYTES = 4 * GIB\nRF_PREFLIGHT_MIN_AVAILABLE_PERCENT = 30.0\nRF_FIT_LIMIT_SECONDS = 1_800.0\nRF_PREDICT_LIMIT_SECONDS = 300.0\nRF_TOTAL_LIMIT_SECONDS = 2_100.0\nRF_MIN_AVAILABLE_BYTES = 2 * GIB\nRF_MIN_AVAILABLE_PERCENT = 15.0\nRF_MONITOR_INTERVAL_SECONDS = 1.0\n\nEXTREME_GROUPS = {\n    "high": lambda y: y >= 300.0,\n    "extreme_high": lambda y: y >= 1000.0,\n    "negative": lambda y: y < 0.0,\n    "very_negative": lambda y: y <= -100.0,\n    "extreme_low": lambda y: y <= -1000.0,\n    "primary_union": lambda y: (y <= -1000.0) | (y >= 1000.0),\n}\nEXTREME_RULE_LABELS = {\n    "high": "Y >= 300",\n    "extreme_high": "Y >= 1000",\n    "negative": "Y < 0",\n    "very_negative": "Y <= -100",\n    "extreme_low": "Y <= -1000",\n    "primary_union": "Y <= -1000 or Y >= 1000",\n}\n\n\ndef package_versions() -> pd.DataFrame:\n    return pd.DataFrame(\n        [\n            {"package": "Python", "version": platform.python_version()},\n            {"package": "NumPy", "version": np.__version__},\n            {"package": "pandas", "version": pd.__version__},\n            {"package": "SciPy", "version": scipy.__version__},\n            {"package": "scikit-learn", "version": sklearn.__version__},\n            {"package": "joblib", "version": joblib.__version__},\n            {"package": "psutil", "version": psutil.__version__},\n        ]\n    )\n\n\ndef safe_skill(model_mae: float, baseline_mae: float) -> float:\n    if not np.isfinite(baseline_mae) or baseline_mae == 0.0:\n        return float("nan")\n    return 1.0 - model_mae / baseline_mae\n\n\ndef overall_metrics(prediction: np.ndarray, target: np.ndarray, baseline: np.ndarray) -> dict[str, float]:\n    error = prediction.astype(np.float64, copy=False) - target.astype(np.float64, copy=False)\n    baseline_error = baseline.astype(np.float64, copy=False) - target.astype(np.float64, copy=False)\n    mae = float(np.mean(np.abs(error), dtype=np.float64))\n    baseline_mae = float(np.mean(np.abs(baseline_error), dtype=np.float64))\n    metrics = {\n        "validation_mae": mae,\n        "validation_rmse": float(np.sqrt(np.mean(np.square(error), dtype=np.float64))),\n        "validation_bias": float(np.mean(error, dtype=np.float64)),\n        "validation_mae_skill": safe_skill(mae, baseline_mae),\n        "baseline_validation_mae": baseline_mae,\n    }\n    del error, baseline_error\n    return metrics\n\n\ndef validate_baseline(\n    frame: pd.DataFrame,\n    training: fe.MatrixBundle,\n    validation: fe.MatrixBundle,\n    representative_checks: pd.DataFrame,\n    chunk_size: int = 8192,\n) -> tuple[np.ndarray, pd.DataFrame]:\n    baseline = validation.F3[:, 38:134]\n    rrp = frame["RRP"].to_numpy(dtype=np.float64, copy=False)\n    maximum_error = 0.0\n    checked = 0\n    for start in range(0, len(validation.X), chunk_size):\n        stop = min(start + chunk_size, len(validation.X))\n        positions = validation.origin_positions[start:stop]\n        expected = rrp[positions[:, None] + fe.HORIZONS[None, :] - fe.HISTORY_STEPS]\n        actual = baseline[start:stop].astype(np.float64)\n        maximum_error = max(maximum_error, float(np.max(np.abs(actual - expected))))\n        checked += expected.size\n        del expected, actual\n\n    previous_catalog = fe.feature_catalog().query("feature_group == \'previous_day\'")\n    representative_ok = bool(\n        (representative_checks["all_96_previous_day_mappings_checked"] == 96).all()\n        and (representative_checks["previous_day_max_abs_error"] <= fe.PRICE_TOLERANCE).all()\n    )\n    summary = pd.DataFrame(\n        [\n            {\n                "check": "baseline_shape",\n                "status": "PASS" if baseline.shape == (52_033, 96) else "FAIL",\n                "evidence": str(baseline.shape),\n            },\n            {\n                "check": "all_validation_baseline_mappings",\n                "status": "PASS" if maximum_error <= fe.PRICE_TOLERANCE else "FAIL",\n                "evidence": f"{checked} mappings; max abs error={maximum_error}",\n            },\n            {\n                "check": "baseline_feature_names",\n                "status": "PASS" if previous_catalog["feature_name"].tolist() == fe.PREVIOUS_DAY_FEATURE_NAMES else "FAIL",\n                "evidence": "prev_day_rrp_h001..h096",\n            },\n            {\n                "check": "baseline_source_offsets",\n                "status": "PASS" if previous_catalog["source_min_offset"].tolist() == list(range(-287, -191)) else "FAIL",\n                "evidence": "t-287..t-192",\n            },\n            {\n                "check": "representative_all_96_baseline_mappings",\n                "status": "PASS" if representative_ok else "FAIL",\n                "evidence": f"{len(representative_checks)} representatives x 96 horizons",\n            },\n            {\n                "check": "baseline_values_finite",\n                "status": "PASS" if np.isfinite(baseline).all() else "FAIL",\n                "evidence": "all validation baseline values",\n            },\n            {\n                "check": "baseline_is_F3_prefix_block_view",\n                "status": "PASS" if np.shares_memory(baseline, validation.F3) else "FAIL",\n                "evidence": "F3 columns 38:134",\n            },\n        ]\n    )\n    return baseline, summary\n\n\ndef _array_hash(value: Any) -> str:\n    array = np.asarray(value)\n    return hashlib.sha256(array.tobytes()).hexdigest()\n\n\ndef _snapshot_scaler(scaler: StandardScaler) -> dict[str, Any]:\n    return {\n        "mean_hash": _array_hash(scaler.mean_),\n        "var_hash": _array_hash(scaler.var_),\n        "scale_hash": _array_hash(scaler.scale_),\n        "n_samples_seen_hash": _array_hash(scaler.n_samples_seen_),\n        "n_samples_seen_value": np.asarray(scaler.n_samples_seen_).tolist(),\n    }\n\n\ndef _candidate_better(candidate: dict, incumbent: dict | None, kind: str) -> bool:\n    if incumbent is None:\n        return True\n    delta = candidate["validation_mae"] - incumbent["validation_mae"]\n    if delta < -TIE_TOLERANCE:\n        return True\n    if abs(delta) > TIE_TOLERANCE:\n        return False\n    if kind == "ridge":\n        return candidate["alpha"] > incumbent["alpha"]\n    candidate_key = (\n        candidate["total_tree_nodes"],\n        candidate["estimated_tree_bytes"],\n        candidate["n_estimators"],\n        candidate["max_depth"],\n        -candidate["min_samples_leaf"],\n        candidate["max_samples"],\n        candidate["candidate_id"],\n    )\n    incumbent_key = (\n        incumbent["total_tree_nodes"],\n        incumbent["estimated_tree_bytes"],\n        incumbent["n_estimators"],\n        incumbent["max_depth"],\n        -incumbent["min_samples_leaf"],\n        incumbent["max_samples"],\n        incumbent["candidate_id"],\n    )\n    return candidate_key < incumbent_key\n\n\ndef run_ridge_experiments(\n    training: fe.MatrixBundle,\n    validation: fe.MatrixBundle,\n    baseline: np.ndarray,\n) -> dict[str, Any]:\n    candidates: list[dict] = []\n    convergence_rows: list[dict] = []\n    scaler_rows: list[dict] = []\n    selected_rows: list[dict] = []\n    selected_f3_prediction: np.ndarray | None = None\n\n    for feature_set in ("F1", "F2", "F3"):\n        feature_slice = fe.FEATURE_SET_SLICES[feature_set]\n        X_train = training.X[:, feature_slice]\n        X_validation = validation.X[:, feature_slice]\n        scaler = StandardScaler(with_mean=True, with_std=True, copy=True)\n        scaler_fit_start = time.perf_counter()\n        scaler.fit(X_train)\n        scaler_fit_seconds = time.perf_counter() - scaler_fit_start\n        before = _snapshot_scaler(scaler)\n\n        X_train_scaled = scaler.transform(X_train)\n        X_validation_scaled = scaler.transform(X_validation)\n        after = _snapshot_scaler(scaler)\n        unchanged = before == after\n        mean_matches = bool(np.allclose(scaler.mean_, X_train.mean(axis=0, dtype=np.float64), rtol=1e-10, atol=1e-10))\n        var_matches = bool(np.allclose(scaler.var_, X_train.var(axis=0, dtype=np.float64), rtol=1e-8, atol=1e-8))\n        seen = np.asarray(scaler.n_samples_seen_)\n        seen_correct = bool(np.all(seen == len(X_train)))\n        scaler_rows.append(\n            {\n                "feature_set": feature_set,\n                "feature_count": X_train.shape[1],\n                "training_rows_seen": int(seen.flat[0]),\n                "scaler_fit_seconds": scaler_fit_seconds,\n                "mean_hash_before": before["mean_hash"],\n                "mean_hash_after": after["mean_hash"],\n                "var_hash_before": before["var_hash"],\n                "var_hash_after": after["var_hash"],\n                "scale_hash_before": before["scale_hash"],\n                "scale_hash_after": after["scale_hash"],\n                "n_samples_seen_hash_before": before["n_samples_seen_hash"],\n                "n_samples_seen_hash_after": after["n_samples_seen_hash"],\n                "attributes_unchanged_after_validation_transform": unchanged,\n                "training_mean_matches": mean_matches,\n                "training_variance_matches": var_matches,\n                "training_row_count_matches": seen_correct,\n                "status": "PASS" if unchanged and mean_matches and var_matches and seen_correct else "FAIL",\n            }\n        )\n        if not scaler_rows[-1]["status"] == "PASS":\n            raise AssertionError(f"Training-only scaler validation failed for {feature_set}.")\n\n        best: dict | None = None\n        best_prediction: np.ndarray | None = None\n        for alpha in RIDGE_ALPHAS:\n            process = psutil.Process(os.getpid())\n            rss_before = process.memory_info().rss\n            model = Ridge(alpha=alpha, **RIDGE_SETTINGS)\n            with warnings.catch_warnings(record=True) as caught:\n                warnings.simplefilter("always")\n                fit_start = time.perf_counter()\n                model.fit(X_train_scaled, training.Y)\n                fit_seconds = time.perf_counter() - fit_start\n            warning_messages = [f"{type(item.message).__name__}: {item.message}" for item in caught]\n            convergence_warning = any(\n                isinstance(item.message, ConvergenceWarning) for item in caught\n            )\n            n_iter = np.asarray(model.n_iter_).reshape(-1) if model.n_iter_ is not None else np.array([], dtype=int)\n            n_iter_complete = len(n_iter) == fe.TARGET_STEPS\n            reached_max = bool(np.any(n_iter >= RIDGE_SETTINGS["max_iter"])) if n_iter_complete else True\n            coefficients_finite = bool(np.isfinite(model.coef_).all())\n            intercept_finite = bool(np.isfinite(model.intercept_).all())\n            predict_start = time.perf_counter()\n            prediction = model.predict(X_validation_scaled)\n            predict_seconds = time.perf_counter() - predict_start\n            prediction_shape_correct = prediction.shape == (52_033, 96)\n            prediction_finite = bool(np.isfinite(prediction).all())\n\n            valid = bool(\n                not convergence_warning\n                and n_iter_complete\n                and not reached_max\n                and coefficients_finite\n                and intercept_finite\n                and prediction_shape_correct\n                and prediction_finite\n            )\n            metrics = overall_metrics(prediction, validation.Y, baseline) if valid else {\n                "validation_mae": float("nan"),\n                "validation_rmse": float("nan"),\n                "validation_bias": float("nan"),\n                "validation_mae_skill": float("nan"),\n                "baseline_validation_mae": float("nan"),\n            }\n            row = {\n                "candidate_id": f"Ridge_{feature_set}_a{int(alpha)}",\n                "model_family": "Ridge",\n                "feature_set": feature_set,\n                "feature_count": X_train.shape[1],\n                "alpha": alpha,\n                "solver": RIDGE_SETTINGS["solver"],\n                "tol": RIDGE_SETTINGS["tol"],\n                "max_iter": RIDGE_SETTINGS["max_iter"],\n                "fit_seconds": fit_seconds,\n                "predict_seconds": predict_seconds,\n                "rss_before_bytes": rss_before,\n                "rss_after_bytes": process.memory_info().rss,\n                "warning_count": len(caught),\n                "convergence_warning": convergence_warning,\n                "n_iter_target_count": len(n_iter),\n                "n_iter_min": int(n_iter.min()) if n_iter_complete else np.nan,\n                "n_iter_max": int(n_iter.max()) if n_iter_complete else np.nan,\n                "n_iter_mean": float(n_iter.mean()) if n_iter_complete else np.nan,\n                "any_target_reached_max_iter": reached_max,\n                "coefficients_finite": coefficients_finite,\n                "intercepts_finite": intercept_finite,\n                "prediction_shape_correct": prediction_shape_correct,\n                "predictions_finite": prediction_finite,\n                "status": "VALID" if valid else "INVALID",\n                "warning_messages": " | ".join(warning_messages),\n                **metrics,\n            }\n            candidates.append(row)\n            for target_index in range(fe.TARGET_STEPS):\n                convergence_rows.append(\n                    {\n                        "candidate_id": row["candidate_id"],\n                        "feature_set": feature_set,\n                        "alpha": alpha,\n                        "target_name": fe.TARGET_NAMES[target_index],\n                        "horizon": target_index + 1,\n                        "n_iter": int(n_iter[target_index]) if n_iter_complete else np.nan,\n                        "reached_max_iter": bool(n_iter[target_index] >= RIDGE_SETTINGS["max_iter"]) if n_iter_complete else True,\n                        "candidate_status": row["status"],\n                    }\n                )\n\n            if valid and _candidate_better(row, best, "ridge"):\n                best = row.copy()\n                if feature_set == "F3":\n                    best_prediction = prediction.astype(np.float32, copy=True)\n            del model, prediction\n            gc.collect()\n\n        if best is None:\n            raise RuntimeError(f"No valid Ridge candidate completed for {feature_set}.")\n        boundary = (\n            "LOWER_BOUNDARY_SELECTION"\n            if best["alpha"] == RIDGE_ALPHAS[0]\n            else "UPPER_BOUNDARY_SELECTION"\n            if best["alpha"] == RIDGE_ALPHAS[-1]\n            else "INTERIOR_SELECTION"\n        )\n        selected_rows.append({**best, "selection_status": boundary})\n        if feature_set == "F3":\n            selected_f3_prediction = best_prediction\n\n        del X_train_scaled, X_validation_scaled, scaler\n        gc.collect()\n\n    if selected_f3_prediction is None:\n        raise RuntimeError("Selected F3 Ridge prediction was not retained for aggregate diagnostics.")\n    return {\n        "candidates": pd.DataFrame(candidates),\n        "convergence": pd.DataFrame(convergence_rows),\n        "scalers": pd.DataFrame(scaler_rows),\n        "selected": pd.DataFrame(selected_rows),\n        "selected_f3_prediction": selected_f3_prediction,\n    }\n\n\ndef _metric_for_mask(prediction: np.ndarray, target: np.ndarray, baseline: np.ndarray, mask: np.ndarray) -> dict[str, float]:\n    if not np.any(mask):\n        return {"mae": float("nan"), "rmse": float("nan"), "bias": float("nan"), "mae_skill": float("nan")}\n    error = prediction[mask].astype(np.float64) - target[mask].astype(np.float64)\n    baseline_error = baseline[mask].astype(np.float64) - target[mask].astype(np.float64)\n    mae = float(np.mean(np.abs(error), dtype=np.float64))\n    baseline_mae = float(np.mean(np.abs(baseline_error), dtype=np.float64))\n    return {\n        "mae": mae,\n        "rmse": float(np.sqrt(np.mean(np.square(error), dtype=np.float64))),\n        "bias": float(np.mean(error, dtype=np.float64)),\n        "mae_skill": safe_skill(mae, baseline_mae),\n    }\n\n\ndef selected_diagnostics(\n    predictions: dict[str, np.ndarray],\n    target: np.ndarray,\n    baseline: np.ndarray,\n    origin_timestamps: np.ndarray,\n) -> dict[str, pd.DataFrame]:\n    overall_rows: list[dict] = []\n    horizon_rows: list[dict] = []\n    extreme_rows: list[dict] = []\n    horizon_extreme_rows: list[dict] = []\n\n    for model_name, prediction in predictions.items():\n        overall_rows.append({"model": model_name, **overall_metrics(prediction, target, baseline)})\n        for horizon_index in range(fe.TARGET_STEPS):\n            mask = np.ones(len(target), dtype=bool)\n            metric = _metric_for_mask(\n                prediction[:, horizon_index], target[:, horizon_index], baseline[:, horizon_index], mask\n            )\n            horizon_rows.append(\n                {\n                    "model": model_name,\n                    "horizon": horizon_index + 1,\n                    "minutes_after_origin": (horizon_index + 1) * 5,\n                    "forecast_target_pair_count": len(target),\n                    "mae": metric["mae"],\n                    "rmse": metric["rmse"],\n                    "bias": metric["bias"],\n                    "mae_skill": metric["mae_skill"],\n                    "status": "AVAILABLE",\n                }\n            )\n\n    for group_name, selector in EXTREME_GROUPS.items():\n        group_mask = selector(target)\n        pair_count = int(group_mask.sum())\n        realised_parts = []\n        for horizon_index in range(fe.TARGET_STEPS):\n            horizon_mask = group_mask[:, horizon_index]\n            if np.any(horizon_mask):\n                realised_parts.append(\n                    origin_timestamps[horizon_mask]\n                    + np.timedelta64((horizon_index + 1) * 5, "m")\n                )\n        unique_timestamps = int(len(np.unique(np.concatenate(realised_parts)))) if realised_parts else 0\n        for model_name, prediction in predictions.items():\n            metric = _metric_for_mask(prediction, target, baseline, group_mask)\n            extreme_rows.append(\n                {\n                    "model": model_name,\n                    "event_group": group_name,\n                    "observed_target_rule": EXTREME_RULE_LABELS[group_name],\n                    "forecast_target_pair_count": pair_count,\n                    "unique_realised_timestamp_count": unique_timestamps,\n                    "mae": metric["mae"],\n                    "rmse": metric["rmse"],\n                    "bias": metric["bias"],\n                    "mae_skill": metric["mae_skill"],\n                    "status": "AVAILABLE" if pair_count else "NO_PAIRS",\n                }\n            )\n\n        for horizon_index in range(fe.TARGET_STEPS):\n            horizon_mask = group_mask[:, horizon_index]\n            horizon_pairs = int(horizon_mask.sum())\n            unique_horizon_times = int(\n                len(\n                    np.unique(\n                        origin_timestamps[horizon_mask]\n                        + np.timedelta64((horizon_index + 1) * 5, "m")\n                    )\n                )\n            )\n            sufficient = horizon_pairs >= 20\n            for model_name, prediction in predictions.items():\n                metric = (\n                    _metric_for_mask(\n                        prediction[:, horizon_index],\n                        target[:, horizon_index],\n                        baseline[:, horizon_index],\n                        horizon_mask,\n                    )\n                    if sufficient\n                    else {"mae": np.nan, "rmse": np.nan, "bias": np.nan, "mae_skill": np.nan}\n                )\n                horizon_extreme_rows.append(\n                    {\n                        "model": model_name,\n                        "event_group": group_name,\n                        "observed_target_rule": EXTREME_RULE_LABELS[group_name],\n                        "horizon": horizon_index + 1,\n                        "minutes_after_origin": (horizon_index + 1) * 5,\n                        "forecast_target_pair_count": horizon_pairs,\n                        "unique_realised_timestamp_count": unique_horizon_times,\n                        "mae": metric["mae"],\n                        "rmse": metric["rmse"],\n                        "bias": metric["bias"],\n                        "mae_skill": metric["mae_skill"],\n                        "status": "AVAILABLE" if sufficient else "INSUFFICIENT",\n                    }\n                )\n\n    return {\n        "overall": pd.DataFrame(overall_rows),\n        "horizon": pd.DataFrame(horizon_rows),\n        "extreme": pd.DataFrame(extreme_rows),\n        "horizon_extreme": pd.DataFrame(horizon_extreme_rows),\n    }\n\n\ndef _forest_size(model: RandomForestRegressor) -> tuple[int, int]:\n    nodes = 0\n    bytes_total = 0\n    attributes = (\n        "children_left",\n        "children_right",\n        "feature",\n        "threshold",\n        "impurity",\n        "n_node_samples",\n        "weighted_n_node_samples",\n        "value",\n    )\n    for estimator in model.estimators_:\n        tree = estimator.tree_\n        nodes += int(tree.node_count)\n        for name in attributes:\n            bytes_total += int(getattr(tree, name).nbytes)\n    return nodes, bytes_total\n\n\ndef write_worker_status(path: Path, phase: str, **extra: Any) -> None:\n    payload = {"phase": phase, "phase_started_monotonic": time.monotonic(), **extra}\n    path.write_text(json.dumps(payload), encoding="utf-8")\n\n\ndef rf_worker_run(\n    workspace: Path,\n    config_id: str,\n    result_path: Path,\n    status_path: Path,\n    include_diagnostics: bool = False,\n) -> None:\n    config = next(item for item in RF_CONFIGS if item["candidate_id"] == config_id)\n    write_worker_status(status_path, "building")\n    data_path = workspace / "data" / "processed" / "prices_NSW1_5min_prepared.csv"\n    checksum_before = fe.sha256_file(data_path)\n    if checksum_before != fe.APPROVED_SHA256:\n        raise RuntimeError("Canonical checksum mismatch inside Random Forest worker.")\n    frame = fe.load_canonical_dataset(data_path)\n    training = fe.build_split_matrices(frame, "training")\n    validation = fe.build_split_matrices(frame, "validation")\n    baseline = validation.F3[:, 38:134]\n\n    settings = {**RF_COMMON, **{key: config[key] for key in ("n_estimators", "max_depth", "min_samples_leaf", "max_samples")}}\n    model = RandomForestRegressor(**settings)\n    write_worker_status(status_path, "fit")\n    fit_start = time.perf_counter()\n    model.fit(training.F3, training.Y)\n    fit_seconds = time.perf_counter() - fit_start\n\n    write_worker_status(status_path, "predict")\n    predict_start = time.perf_counter()\n    prediction = model.predict(validation.F3)\n    predict_seconds = time.perf_counter() - predict_start\n    if prediction.shape != (52_033, 96) or not np.isfinite(prediction).all():\n        raise RuntimeError("Random Forest produced an invalid validation prediction matrix.")\n\n    write_worker_status(status_path, "metrics")\n    metrics = overall_metrics(prediction, validation.Y, baseline)\n    diagnostics = (\n        selected_diagnostics(\n            {config_id: prediction}, validation.Y, baseline, validation.origin_timestamps\n        )\n        if include_diagnostics\n        else None\n    )\n    total_nodes, tree_bytes = _forest_size(model)\n    checksum_after = fe.sha256_file(data_path)\n    result = {\n        "candidate": {\n            **config,\n            **RF_COMMON,\n            "fit_seconds": fit_seconds,\n            "predict_seconds": predict_seconds,\n            "total_tree_nodes": total_nodes,\n            "estimated_tree_bytes": tree_bytes,\n            "prediction_shape": list(prediction.shape),\n            "predictions_finite": True,\n            "canonical_checksum_before": checksum_before,\n            "canonical_checksum_after": checksum_after,\n            "status": "VALID",\n            **metrics,\n        },\n        "diagnostics": (\n            {name: table.to_dict(orient="records") for name, table in diagnostics.items()}\n            if diagnostics is not None\n            else None\n        ),\n        "diagnostic_scope": "selected_candidate_only" if include_diagnostics else "overall_metrics_only",\n    }\n    result_path.write_text(json.dumps(result, allow_nan=True), encoding="utf-8")\n    write_worker_status(status_path, "complete")\n\n\ndef _process_tree_rss(process: psutil.Process) -> int:\n    processes = [process]\n    try:\n        processes.extend(process.children(recursive=True))\n    except (psutil.NoSuchProcess, psutil.AccessDenied):\n        pass\n    total = 0\n    for item in processes:\n        try:\n            total += item.memory_info().rss\n        except (psutil.NoSuchProcess, psutil.AccessDenied):\n            pass\n    return total\n\n\ndef _terminate_process_tree(process: psutil.Process) -> None:\n    try:\n        children = process.children(recursive=True)\n    except (psutil.NoSuchProcess, psutil.AccessDenied):\n        children = []\n    for item in reversed(children):\n        try:\n            item.terminate()\n        except (psutil.NoSuchProcess, psutil.AccessDenied):\n            pass\n    try:\n        process.terminate()\n    except (psutil.NoSuchProcess, psutil.AccessDenied):\n        pass\n    _, alive = psutil.wait_procs(children + [process], timeout=10)\n    for item in alive:\n        try:\n            item.kill()\n        except (psutil.NoSuchProcess, psutil.AccessDenied):\n            pass\n\n\ndef _read_status(path: Path) -> dict[str, Any]:\n    if not path.exists():\n        return {"phase": "starting", "phase_started_monotonic": time.monotonic()}\n    try:\n        return json.loads(path.read_text(encoding="utf-8"))\n    except json.JSONDecodeError:\n        return {"phase": "status_write_in_progress", "phase_started_monotonic": time.monotonic()}\n\n\ndef run_monitored_rf_candidate(\n    workspace: Path,\n    config: dict[str, Any],\n    include_diagnostics: bool = False,\n) -> dict[str, Any]:\n    total_memory = psutil.virtual_memory().total\n    rss_limit = int(min(12 * GIB, 0.70 * total_memory))\n    with tempfile.TemporaryDirectory(prefix=f"phase6_{config[\'candidate_id\']}_") as temp_dir:\n        temp = Path(temp_dir)\n        result_path = temp / "result.json"\n        status_path = temp / "status.json"\n        worker_path = workspace / "code" / "phase6_rf_worker.py"\n        command = [\n            sys.executable,\n            str(worker_path),\n            "--workspace",\n            str(workspace),\n            "--candidate-id",\n            config["candidate_id"],\n            "--result-path",\n            str(result_path),\n            "--status-path",\n            str(status_path),\n        ]\n        if include_diagnostics:\n            command.append("--include-diagnostics")\n        started = time.monotonic()\n        popen = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)\n        process = psutil.Process(popen.pid)\n        peak_rss = 0\n        minimum_available_bytes = psutil.virtual_memory().available\n        minimum_available_percent = 100.0 * minimum_available_bytes / total_memory\n        failure_reason = None\n        failure_observed = None\n\n        while popen.poll() is None:\n            now = time.monotonic()\n            status = _read_status(status_path)\n            phase = status.get("phase", "starting")\n            phase_elapsed = now - float(status.get("phase_started_monotonic", now))\n            rss = _process_tree_rss(process)\n            memory = psutil.virtual_memory()\n            available_percent = 100.0 * memory.available / memory.total\n            peak_rss = max(peak_rss, rss)\n            minimum_available_bytes = min(minimum_available_bytes, memory.available)\n            minimum_available_percent = min(minimum_available_percent, available_percent)\n\n            if rss > rss_limit:\n                failure_reason, failure_observed = "process_tree_peak_rss", rss\n            elif memory.available < RF_MIN_AVAILABLE_BYTES:\n                failure_reason, failure_observed = "system_available_bytes", memory.available\n            elif available_percent < RF_MIN_AVAILABLE_PERCENT:\n                failure_reason, failure_observed = "system_available_percent", available_percent\n            elif now - started > RF_TOTAL_LIMIT_SECONDS:\n                failure_reason, failure_observed = "total_seconds", now - started\n            elif phase == "fit" and phase_elapsed > RF_FIT_LIMIT_SECONDS:\n                failure_reason, failure_observed = "fit_seconds", phase_elapsed\n            elif phase == "predict" and phase_elapsed > RF_PREDICT_LIMIT_SECONDS:\n                failure_reason, failure_observed = "predict_seconds", phase_elapsed\n\n            if failure_reason is not None:\n                _terminate_process_tree(process)\n                break\n            time.sleep(RF_MONITOR_INTERVAL_SECONDS)\n\n        stdout, stderr = popen.communicate(timeout=30)\n        elapsed = time.monotonic() - started\n        monitoring = {\n            "candidate_id": config["candidate_id"],\n            "run_kind": "selected_diagnostics" if include_diagnostics else "candidate_selection",\n            "isolated_worker": True,\n            "monitor_interval_seconds": RF_MONITOR_INTERVAL_SECONDS,\n            "peak_process_tree_rss_bytes": peak_rss,\n            "rss_limit_bytes": rss_limit,\n            "minimum_available_memory_bytes": minimum_available_bytes,\n            "minimum_available_memory_percent": minimum_available_percent,\n            "minimum_allowed_available_bytes": RF_MIN_AVAILABLE_BYTES,\n            "minimum_allowed_available_percent": RF_MIN_AVAILABLE_PERCENT,\n            "elapsed_total_seconds": elapsed,\n            "fit_limit_seconds": RF_FIT_LIMIT_SECONDS,\n            "predict_limit_seconds": RF_PREDICT_LIMIT_SECONDS,\n            "total_limit_seconds": RF_TOTAL_LIMIT_SECONDS,\n            "worker_return_code": popen.returncode,\n            "resource_failure_reason": failure_reason or "",\n            "resource_failure_observed": failure_observed if failure_observed is not None else np.nan,\n        }\n        if failure_reason is not None:\n            return {\n                "candidate": {**config, **RF_COMMON, "status": "RESOURCE_FAILED"},\n                "monitoring": monitoring,\n                "diagnostics": None,\n                "stdout": stdout,\n                "stderr": stderr,\n            }\n        if popen.returncode != 0 or not result_path.exists():\n            raise RuntimeError(\n                f"Random Forest worker {config[\'candidate_id\']} failed outside the approved resource rule. "\n                f"Return code={popen.returncode}; stderr={stderr[-4000:]}"\n            )\n        result = json.loads(result_path.read_text(encoding="utf-8"))\n        result["monitoring"] = monitoring\n        result["stdout"] = stdout\n        result["stderr"] = stderr\n        return result\n\n\ndef run_random_forest_search(workspace: Path) -> dict[str, Any]:\n    memory = psutil.virtual_memory()\n    available_percent = 100.0 * memory.available / memory.total\n    preflight = {\n        "available_memory_bytes": memory.available,\n        "available_memory_percent": available_percent,\n        "required_available_bytes": RF_PREFLIGHT_MIN_AVAILABLE_BYTES,\n        "required_available_percent": RF_PREFLIGHT_MIN_AVAILABLE_PERCENT,\n        "status": "PASS"\n        if memory.available >= RF_PREFLIGHT_MIN_AVAILABLE_BYTES\n        and available_percent >= RF_PREFLIGHT_MIN_AVAILABLE_PERCENT\n        else "FAIL",\n    }\n    candidate_rows: list[dict] = []\n    monitoring_rows: list[dict] = []\n    successful_results: list[dict] = []\n    stopped = False\n\n    if preflight["status"] == "FAIL":\n        for index, config in enumerate(RF_CONFIGS):\n            candidate_rows.append(\n                {\n                    **config,\n                    **RF_COMMON,\n                    "status": "RESOURCE_FAILED" if index == 0 else "NOT_RUN_AFTER_RESOURCE_FAILURE",\n                    "resource_failure_reason": "preflight_available_memory" if index == 0 else "prior_resource_failure",\n                    "validation_mae": np.nan,\n                }\n            )\n        return {\n            "preflight": pd.DataFrame([preflight]),\n            "candidates": pd.DataFrame(candidate_rows),\n            "monitoring": pd.DataFrame(),\n            "selected": None,\n            "selected_diagnostics": None,\n            "diagnostics_status": "NOT_RUN",\n            "candidate_diagnostics_calculated": False,\n            "search_status": "RESOURCE_FAILED",\n        }\n\n    for config in RF_CONFIGS:\n        if stopped:\n            candidate_rows.append(\n                {\n                    **config,\n                    **RF_COMMON,\n                    "status": "NOT_RUN_AFTER_RESOURCE_FAILURE",\n                    "resource_failure_reason": "prior_resource_failure",\n                    "validation_mae": np.nan,\n                }\n            )\n            continue\n        result = run_monitored_rf_candidate(workspace, config, include_diagnostics=False)\n        monitoring_rows.append(result["monitoring"])\n        candidate = {**result["candidate"], **result["monitoring"]}\n        candidate_rows.append(candidate)\n        if candidate["status"] == "RESOURCE_FAILED":\n            stopped = True\n        else:\n            successful_results.append(result)\n\n    if not successful_results:\n        search_status = "RESOURCE_FAILED"\n        selected = None\n        selected_diagnostics = None\n        diagnostics_status = "NOT_RUN"\n    else:\n        search_status = "RESOURCE_CONSTRAINED" if stopped else "COMPLETE"\n        selected_result: dict | None = None\n        for result in successful_results:\n            if selected_result is None or _candidate_better(\n                result["candidate"], selected_result["candidate"], "rf"\n            ):\n                selected_result = result\n        selected = selected_result["candidate"]\n        selected_config = next(\n            item for item in RF_CONFIGS if item["candidate_id"] == selected["candidate_id"]\n        )\n        diagnostic_result = run_monitored_rf_candidate(\n            workspace, selected_config, include_diagnostics=True\n        )\n        monitoring_rows.append(diagnostic_result["monitoring"])\n        if diagnostic_result["candidate"]["status"] == "RESOURCE_FAILED":\n            diagnostics_status = "RESOURCE_FAILED"\n            selected_diagnostics = None\n            search_status = "RESOURCE_CONSTRAINED"\n        else:\n            diagnostics_status = "COMPLETE"\n            selected_diagnostics = {\n                name: pd.DataFrame(rows)\n                for name, rows in diagnostic_result["diagnostics"].items()\n            }\n\n    return {\n        "preflight": pd.DataFrame([preflight]),\n        "candidates": pd.DataFrame(candidate_rows),\n        "monitoring": pd.DataFrame(monitoring_rows),\n        "selected": selected,\n        "selected_diagnostics": selected_diagnostics,\n        "diagnostics_status": diagnostics_status,\n        "candidate_diagnostics_calculated": False,\n        "search_status": search_status,\n    }\n', 'phase6_rf_worker.py': '"""Isolated Random Forest worker used by the Phase 6 resource monitor."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--workspace", required=True)\n    parser.add_argument("--candidate-id", required=True)\n    parser.add_argument("--result-path", required=True)\n    parser.add_argument("--status-path", required=True)\n    parser.add_argument("--include-diagnostics", action="store_true")\n    args = parser.parse_args()\n\n    workspace = Path(args.workspace).resolve()\n    code_dir = workspace / "code"\n    if str(code_dir) not in sys.path:\n        sys.path.insert(0, str(code_dir))\n\n    import phase6_experiments as experiments\n\n    experiments.rf_worker_run(\n        workspace=workspace,\n        config_id=args.candidate_id,\n        result_path=Path(args.result_path),\n        status_path=Path(args.status_path),\n        include_diagnostics=args.include_diagnostics,\n    )\n\n\nif __name__ == "__main__":\n    main()\n', 'phase7_evaluation.py': '"""Frozen Phase 7 development refit and held-out test evaluation.\n\nThis module keeps test evaluation separate from model selection. It returns compact\naggregate evidence only and never persists matrices, predictions, scalers, or fitted\nmodels.\n"""\n\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport warnings\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nimport psutil\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import Ridge\nfrom sklearn.preprocessing import StandardScaler\n\nimport feature_engineering as fe\nimport phase6_experiments as p6\n\n\nRIDGE_ALPHA = 1_000.0\nRIDGE_SETTINGS = {\n    "fit_intercept": True,\n    "solver": "lsqr",\n    "tol": 1e-4,\n    "max_iter": 5_000,\n    "copy_X": True,\n}\nSCALER_SETTINGS = {"with_mean": True, "with_std": True, "copy": True}\nRF02_CONFIG = {\n    "candidate_id": "RF02",\n    "n_estimators": 40,\n    "max_depth": 16,\n    "min_samples_leaf": 20,\n    "max_samples": 0.25,\n    "criterion": "squared_error",\n    "min_samples_split": 2,\n    "max_features": 0.5,\n    "bootstrap": True,\n    "oob_score": False,\n    "warm_start": False,\n    "random_state": 42,\n    "n_jobs": -1,\n    "verbose": 0,\n    "ccp_alpha": 0.0,\n}\n\nEXPECTED_DEVELOPMENT_RECORDS = 289_140\nEXPECTED_TEST_RECORDS = 59_231\nEXPECTED_DEVELOPMENT_WINDOWS = 287_991\nEXPECTED_TEST_WINDOWS = 58_848\nEXPECTED_ADDITIONAL_DEVELOPMENT_WINDOWS = 383\nEXPECTED_TEST_FIRST_ORIGIN = pd.Timestamp("2024-07-01 23:55:00")\nEXPECTED_TEST_LAST_ORIGIN = pd.Timestamp("2025-01-22 07:50:00")\nEXPECTED_PRIMARY_UNION_PAIRS = 22_492\nEXPECTED_PRIMARY_UNION_TIMESTAMPS = 235\n\nGIB = 1024**3\nRF_PREFLIGHT_MIN_AVAILABLE_BYTES = 4 * GIB\nRF_PREFLIGHT_MIN_AVAILABLE_PERCENT = 30.0\nRF_MIN_AVAILABLE_BYTES = 2 * GIB\nRF_MIN_AVAILABLE_PERCENT = 15.0\nRF_FIT_LIMIT_SECONDS = 1_800.0\nRF_PREDICT_LIMIT_SECONDS = 300.0\nRF_TOTAL_LIMIT_SECONDS = 2_100.0\nRF_MONITOR_INTERVAL_SECONDS = 1.0\n\n\ndef package_versions() -> pd.DataFrame:\n    return p6.package_versions()\n\n\ndef _array_hash(value: Any) -> str:\n    return hashlib.sha256(np.asarray(value).tobytes()).hexdigest()\n\n\ndef _snapshot_scaler(scaler: StandardScaler) -> dict[str, Any]:\n    return {\n        "mean_hash": _array_hash(scaler.mean_),\n        "var_hash": _array_hash(scaler.var_),\n        "scale_hash": _array_hash(scaler.scale_),\n        "n_samples_seen_hash": _array_hash(scaler.n_samples_seen_),\n        "n_samples_seen": int(np.asarray(scaler.n_samples_seen_).reshape(-1)[0]),\n    }\n\n\ndef period_record_summary(frame: pd.DataFrame) -> pd.DataFrame:\n    rows = []\n    for period in ("development", "test"):\n        start, end = fe.SPLITS[period]\n        part = frame.loc[(frame["date_time"] >= start) & (frame["date_time"] <= end)]\n        rows.append(\n            {\n                "period": period,\n                "inclusive_start": start,\n                "inclusive_end": end,\n                "record_count": len(part),\n                "continuity_segment_count": int(part["continuity_segment_id"].nunique()),\n                "first_timestamp": part["date_time"].iloc[0],\n                "last_timestamp": part["date_time"].iloc[-1],\n                "expected_window_count": fe.EXPECTED_SAMPLE_COUNTS[period],\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef mapping_validation(\n    frame: pd.DataFrame,\n    development: fe.MatrixBundle,\n    test: fe.MatrixBundle,\n) -> pd.DataFrame:\n    """Independently validate complete feature/target mappings for compact samples."""\n\n    selections: list[tuple[str, int, str]] = []\n    for period, bundle in (("development", development), ("test", test)):\n        for segment_id in np.unique(bundle.segment_ids):\n            indices = np.flatnonzero(bundle.segment_ids == segment_id)\n            selections.append((period, int(indices[0]), f"first origin in segment {segment_id}"))\n            selections.append((period, int(indices[-1]), f"last origin in segment {segment_id}"))\n\n    rng = np.random.default_rng(fe.RANDOM_SEED)\n    for period, bundle in (("development", development), ("test", test)):\n        for ordinal, row_index in enumerate(\n            rng.choice(len(bundle.X), size=4, replace=False), start=1\n        ):\n            selections.append((period, int(row_index), f"seed-42 sample {ordinal}"))\n\n    rows: list[dict[str, Any]] = []\n    bundles = {"development": development, "test": test}\n    for representative_id, (period, row_index, reason) in enumerate(selections, start=1):\n        bundle = bundles[period]\n        position = int(bundle.origin_positions[row_index])\n        expected_x, details = fe._expected_feature_vector(frame, position)\n        expected_y = fe._expected_target(frame, position)\n        actual_x = bundle.X[row_index].astype(np.float64)\n        actual_y = bundle.Y[row_index].astype(np.float64)\n        lag_error = float(np.max(np.abs(actual_x[0:12] - expected_x[0:12])))\n        calendar_error = float(np.max(np.abs(actual_x[12:18] - expected_x[12:18])))\n        rolling_error = float(np.max(np.abs(actual_x[18:38] - expected_x[18:38])))\n        previous_error = float(np.max(np.abs(actual_x[38:134] - details["previous"])))\n        target_error = float(np.max(np.abs(actual_y - expected_y)))\n        origin = pd.Timestamp(bundle.origin_timestamps[row_index])\n        period_start, period_end = fe.SPLITS[period]\n        segment = int(bundle.segment_ids[row_index])\n        history_start = position - fe.HISTORY_STEPS + 1\n        target_end = position + fe.TARGET_STEPS\n        contained = bool(\n            int(frame.iloc[history_start]["continuity_segment_id"]) == segment\n            and int(frame.iloc[target_end]["continuity_segment_id"]) == segment\n            and period_start <= pd.Timestamp(frame.iloc[history_start]["date_time"])\n            and pd.Timestamp(frame.iloc[target_end]["date_time"]) <= period_end\n        )\n        no_future_feature = bool(fe.feature_catalog()["source_max_offset"].le(0).all())\n        status = "PASS" if (\n            lag_error <= fe.PRICE_TOLERANCE\n            and calendar_error <= fe.CALENDAR_TOLERANCE\n            and rolling_error <= fe.PRICE_TOLERANCE\n            and previous_error <= fe.PRICE_TOLERANCE\n            and target_error <= fe.PRICE_TOLERANCE\n            and contained\n            and no_future_feature\n            and pd.Timestamp(frame.iloc[position]["date_time"]) == origin\n        ) else "FAIL"\n        rows.append(\n            {\n                "representative_id": representative_id,\n                "period": period,\n                "selection_reason": reason,\n                "row_index": row_index,\n                "continuity_segment_id": segment,\n                "forecast_origin": origin,\n                "all_12_recent_lags_checked": 12,\n                "all_6_calendar_features_checked": 6,\n                "all_20_rolling_features_checked": 20,\n                "all_96_previous_day_mappings_checked": 96,\n                "all_96_target_mappings_checked": 96,\n                "recent_lag_max_abs_error": lag_error,\n                "calendar_max_abs_error": calendar_error,\n                "rolling_max_abs_error": rolling_error,\n                "previous_day_max_abs_error": previous_error,\n                "target_max_abs_error": target_error,\n                "window_contained": contained,\n                "no_feature_after_origin": no_future_feature,\n                "status": status,\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef validate_test_baseline(\n    frame: pd.DataFrame,\n    test: fe.MatrixBundle,\n    representatives: pd.DataFrame,\n    chunk_size: int = 8192,\n) -> tuple[np.ndarray, pd.DataFrame]:\n    baseline = test.F3[:, 38:134]\n    rrp = frame["RRP"].to_numpy(dtype=np.float64, copy=False)\n    maximum_error = 0.0\n    checked = 0\n    for start in range(0, len(test.X), chunk_size):\n        stop = min(start + chunk_size, len(test.X))\n        positions = test.origin_positions[start:stop]\n        expected = rrp[positions[:, None] + fe.HORIZONS[None, :] - fe.HISTORY_STEPS]\n        actual = baseline[start:stop].astype(np.float64)\n        maximum_error = max(maximum_error, float(np.max(np.abs(actual - expected))))\n        checked += expected.size\n        del expected, actual\n\n    test_representatives = representatives.loc[representatives["period"] == "test"]\n    summary = pd.DataFrame(\n        [\n            {\n                "check": "baseline_shape",\n                "status": "PASS" if baseline.shape == (EXPECTED_TEST_WINDOWS, 96) else "FAIL",\n                "evidence": str(baseline.shape),\n            },\n            {\n                "check": "all_test_baseline_mappings",\n                "status": "PASS" if maximum_error <= fe.PRICE_TOLERANCE else "FAIL",\n                "evidence": f"{checked} mappings; max abs error={maximum_error}",\n            },\n            {\n                "check": "representative_all_96_baseline_mappings",\n                "status": "PASS"\n                if (\n                    test_representatives["all_96_previous_day_mappings_checked"].eq(96).all()\n                    and test_representatives["previous_day_max_abs_error"].le(fe.PRICE_TOLERANCE).all()\n                )\n                else "FAIL",\n                "evidence": f"{len(test_representatives)} test representatives x 96 horizons",\n            },\n            {\n                "check": "baseline_feature_names",\n                "status": "PASS"\n                if fe.FEATURE_NAMES[38:134] == fe.PREVIOUS_DAY_FEATURE_NAMES\n                else "FAIL",\n                "evidence": "prev_day_rrp_h001..h096",\n            },\n            {\n                "check": "baseline_values_finite",\n                "status": "PASS" if np.isfinite(baseline).all() else "FAIL",\n                "evidence": "all test baseline values",\n            },\n            {\n                "check": "baseline_is_F3_previous_day_view",\n                "status": "PASS" if np.shares_memory(baseline, test.F3) else "FAIL",\n                "evidence": "F3 columns 38:134",\n            },\n        ]\n    )\n    return baseline, summary\n\n\ndef fit_final_ridge(\n    development: fe.MatrixBundle,\n    test: fe.MatrixBundle,\n    baseline: np.ndarray,\n) -> dict[str, Any]:\n    X_development = development.F3\n    X_test = test.F3\n    scaler = StandardScaler(**SCALER_SETTINGS)\n    scaler_fit_start = time.perf_counter()\n    scaler.fit(X_development)\n    scaler_fit_seconds = time.perf_counter() - scaler_fit_start\n    before = _snapshot_scaler(scaler)\n\n    X_development_scaled = scaler.transform(X_development)\n    X_test_scaled = scaler.transform(X_test)\n    after = _snapshot_scaler(scaler)\n    mean_matches = bool(\n        np.allclose(\n            scaler.mean_, X_development.mean(axis=0, dtype=np.float64), rtol=1e-10, atol=1e-10\n        )\n    )\n    var_matches = bool(\n        np.allclose(\n            scaler.var_, X_development.var(axis=0, dtype=np.float64), rtol=1e-8, atol=1e-8\n        )\n    )\n    scaler_valid = bool(\n        before == after\n        and before["n_samples_seen"] == EXPECTED_DEVELOPMENT_WINDOWS\n        and mean_matches\n        and var_matches\n    )\n    scaler_summary = pd.DataFrame(\n        [\n            {\n                **SCALER_SETTINGS,\n                "feature_set": "F3",\n                "feature_count": X_development.shape[1],\n                "development_rows_seen": before["n_samples_seen"],\n                "scaler_fit_seconds": scaler_fit_seconds,\n                "mean_hash_before": before["mean_hash"],\n                "mean_hash_after": after["mean_hash"],\n                "var_hash_before": before["var_hash"],\n                "var_hash_after": after["var_hash"],\n                "scale_hash_before": before["scale_hash"],\n                "scale_hash_after": after["scale_hash"],\n                "n_samples_seen_hash_before": before["n_samples_seen_hash"],\n                "n_samples_seen_hash_after": after["n_samples_seen_hash"],\n                "attributes_unchanged_after_test_transform": before == after,\n                "development_mean_matches": mean_matches,\n                "development_variance_matches": var_matches,\n                "status": "PASS" if scaler_valid else "FAIL",\n            }\n        ]\n    )\n    if not scaler_valid:\n        raise AssertionError("Development-only Ridge scaler validation failed.")\n\n    model = Ridge(alpha=RIDGE_ALPHA, **RIDGE_SETTINGS)\n    with warnings.catch_warnings(record=True) as caught:\n        warnings.simplefilter("always")\n        fit_start = time.perf_counter()\n        model.fit(X_development_scaled, development.Y)\n        fit_seconds = time.perf_counter() - fit_start\n    warning_messages = [f"{type(item.message).__name__}: {item.message}" for item in caught]\n    convergence_warning = any(isinstance(item.message, ConvergenceWarning) for item in caught)\n    n_iter = (\n        np.asarray(model.n_iter_).reshape(-1)\n        if model.n_iter_ is not None\n        else np.array([], dtype=int)\n    )\n    n_iter_complete = len(n_iter) == fe.TARGET_STEPS\n    reached_max = bool(np.any(n_iter >= RIDGE_SETTINGS["max_iter"])) if n_iter_complete else True\n    coefficient_finite = bool(np.isfinite(model.coef_).all())\n    intercept_finite = bool(np.isfinite(model.intercept_).all())\n    predict_start = time.perf_counter()\n    prediction = model.predict(X_test_scaled).astype(np.float32, copy=False)\n    predict_seconds = time.perf_counter() - predict_start\n    prediction_valid = bool(\n        prediction.shape == (EXPECTED_TEST_WINDOWS, fe.TARGET_STEPS)\n        and np.isfinite(prediction).all()\n    )\n    valid = bool(\n        not convergence_warning\n        and n_iter_complete\n        and not reached_max\n        and coefficient_finite\n        and intercept_finite\n        and prediction_valid\n    )\n    if not valid:\n        raise AssertionError("Final Ridge refit or prediction failed numerical validation.")\n\n    convergence = pd.DataFrame(\n        {\n            "target_name": fe.TARGET_NAMES,\n            "horizon": fe.HORIZONS,\n            "n_iter": n_iter,\n            "reached_max_iter": n_iter >= RIDGE_SETTINGS["max_iter"],\n        }\n    )\n    summary = pd.DataFrame(\n        [\n            {\n                "model": "Ridge F3 (alpha=1000)",\n                "feature_set": "F3",\n                "alpha": RIDGE_ALPHA,\n                **RIDGE_SETTINGS,\n                "fit_seconds": fit_seconds,\n                "predict_seconds": predict_seconds,\n                "warning_count": len(caught),\n                "warning_messages": " | ".join(warning_messages),\n                "convergence_warning": convergence_warning,\n                "n_iter_target_count": len(n_iter),\n                "n_iter_min": int(n_iter.min()),\n                "n_iter_max": int(n_iter.max()),\n                "any_target_reached_max_iter": reached_max,\n                "coefficients_finite": coefficient_finite,\n                "intercepts_finite": intercept_finite,\n                "prediction_shape": str(prediction.shape),\n                "predictions_finite": bool(np.isfinite(prediction).all()),\n                "status": "VALID",\n                **test_metrics(prediction, test.Y, baseline),\n            }\n        ]\n    )\n\n    del model, scaler, X_development_scaled, X_test_scaled\n    gc.collect()\n    return {\n        "prediction": prediction,\n        "scaler": scaler_summary,\n        "convergence": convergence,\n        "summary": summary,\n    }\n\n\ndef test_metrics(\n    prediction: np.ndarray, target: np.ndarray, baseline: np.ndarray\n) -> dict[str, float]:\n    metrics = p6.overall_metrics(prediction, target, baseline)\n    return {\n        "test_mae": metrics["validation_mae"],\n        "test_rmse": metrics["validation_rmse"],\n        "test_bias": metrics["validation_bias"],\n        "test_mae_skill": metrics["validation_mae_skill"],\n        "baseline_test_mae": metrics["baseline_validation_mae"],\n    }\n\n\ndef selected_test_diagnostics(\n    predictions: dict[str, np.ndarray],\n    target: np.ndarray,\n    baseline: np.ndarray,\n    origin_timestamps: np.ndarray,\n) -> dict[str, pd.DataFrame]:\n    result = p6.selected_diagnostics(predictions, target, baseline, origin_timestamps)\n    result["overall"] = result["overall"].rename(\n        columns={\n            "validation_mae": "test_mae",\n            "validation_rmse": "test_rmse",\n            "validation_bias": "test_bias",\n            "validation_mae_skill": "test_mae_skill",\n            "baseline_validation_mae": "baseline_test_mae",\n        }\n    )\n    return result\n\n\ndef _forest_size(model: RandomForestRegressor) -> tuple[int, int]:\n    return p6._forest_size(model)\n\n\ndef _write_status(path: Path, phase: str) -> None:\n    p6.write_worker_status(path, phase)\n\n\ndef rf02_worker_run(\n    workspace: Path,\n    result_path: Path,\n    status_path: Path,\n) -> None:\n    _write_status(status_path, "building")\n    data_path = workspace / "data" / "processed" / "prices_NSW1_5min_prepared.csv"\n    checksum_before = fe.sha256_file(data_path)\n    if checksum_before != fe.APPROVED_SHA256:\n        raise RuntimeError("Canonical checksum mismatch inside Phase 7 RF02 worker.")\n    frame = fe.load_canonical_dataset(data_path)\n    development = fe.build_split_matrices(frame, "development")\n    test = fe.build_split_matrices(frame, "test")\n    baseline = test.F3[:, 38:134]\n\n    model = RandomForestRegressor(**{k: v for k, v in RF02_CONFIG.items() if k != "candidate_id"})\n    _write_status(status_path, "fit")\n    fit_start = time.perf_counter()\n    model.fit(development.F3, development.Y)\n    fit_seconds = time.perf_counter() - fit_start\n\n    _write_status(status_path, "predict")\n    predict_start = time.perf_counter()\n    prediction = model.predict(test.F3).astype(np.float32, copy=False)\n    predict_seconds = time.perf_counter() - predict_start\n    if prediction.shape != (EXPECTED_TEST_WINDOWS, fe.TARGET_STEPS):\n        raise RuntimeError(f"RF02 prediction shape is {prediction.shape}.")\n    if not np.isfinite(prediction).all():\n        raise RuntimeError("RF02 produced non-finite test predictions.")\n\n    _write_status(status_path, "metrics")\n    diagnostics = selected_test_diagnostics(\n        {"Random Forest (RF02)": prediction}, test.Y, baseline, test.origin_timestamps\n    )\n    nodes, tree_bytes = _forest_size(model)\n    checksum_after = fe.sha256_file(data_path)\n    result = {\n        "status": "VALID",\n        "configuration": RF02_CONFIG,\n        "fit_seconds": fit_seconds,\n        "predict_seconds": predict_seconds,\n        "total_tree_nodes": nodes,\n        "estimated_tree_bytes": tree_bytes,\n        "prediction_shape": list(prediction.shape),\n        "predictions_finite": True,\n        "canonical_checksum_before": checksum_before,\n        "canonical_checksum_after": checksum_after,\n        "diagnostics_scope": "selected_RF02_test_aggregates_only",\n        "diagnostics": {\n            name: table.to_dict(orient="records") for name, table in diagnostics.items()\n        },\n    }\n    result_path.write_text(json.dumps(result, allow_nan=True), encoding="utf-8")\n    _write_status(status_path, "complete")\n\n\ndef run_monitored_rf02(workspace: Path) -> dict[str, Any]:\n    memory = psutil.virtual_memory()\n    available_percent = 100.0 * memory.available / memory.total\n    preflight = {\n        "available_memory_bytes": memory.available,\n        "available_memory_percent": available_percent,\n        "required_available_bytes": RF_PREFLIGHT_MIN_AVAILABLE_BYTES,\n        "required_available_percent": RF_PREFLIGHT_MIN_AVAILABLE_PERCENT,\n        "status": "PASS"\n        if memory.available >= RF_PREFLIGHT_MIN_AVAILABLE_BYTES\n        and available_percent >= RF_PREFLIGHT_MIN_AVAILABLE_PERCENT\n        else "FAIL",\n    }\n    if preflight["status"] == "FAIL":\n        return {\n            "status": "RESOURCE_FAILED",\n            "preflight": pd.DataFrame([preflight]),\n            "monitoring": pd.DataFrame(),\n            "summary": pd.DataFrame(),\n            "diagnostics": None,\n        }\n\n    total_memory = memory.total\n    rss_limit = int(min(12 * GIB, 0.70 * total_memory))\n    with tempfile.TemporaryDirectory(prefix="phase7_RF02_") as temp_dir:\n        temp = Path(temp_dir)\n        result_path = temp / "result.json"\n        status_path = temp / "status.json"\n        worker_path = workspace / "code" / "phase7_rf_worker.py"\n        command = [\n            sys.executable,\n            str(worker_path),\n            "--workspace",\n            str(workspace),\n            "--result-path",\n            str(result_path),\n            "--status-path",\n            str(status_path),\n        ]\n        started = time.monotonic()\n        popen = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)\n        process = psutil.Process(popen.pid)\n        peak_rss = 0\n        minimum_available_bytes = psutil.virtual_memory().available\n        minimum_available_percent = 100.0 * minimum_available_bytes / total_memory\n        failure_reason = None\n        failure_observed: float | int | None = None\n\n        while popen.poll() is None:\n            now = time.monotonic()\n            status = p6._read_status(status_path)\n            phase = status.get("phase", "starting")\n            phase_elapsed = now - float(status.get("phase_started_monotonic", now))\n            rss = p6._process_tree_rss(process)\n            current_memory = psutil.virtual_memory()\n            current_percent = 100.0 * current_memory.available / current_memory.total\n            peak_rss = max(peak_rss, rss)\n            minimum_available_bytes = min(minimum_available_bytes, current_memory.available)\n            minimum_available_percent = min(minimum_available_percent, current_percent)\n\n            if rss > rss_limit:\n                failure_reason, failure_observed = "process_tree_peak_rss", rss\n            elif current_memory.available < RF_MIN_AVAILABLE_BYTES:\n                failure_reason, failure_observed = "system_available_bytes", current_memory.available\n            elif current_percent < RF_MIN_AVAILABLE_PERCENT:\n                failure_reason, failure_observed = "system_available_percent", current_percent\n            elif now - started > RF_TOTAL_LIMIT_SECONDS:\n                failure_reason, failure_observed = "total_seconds", now - started\n            elif phase == "fit" and phase_elapsed > RF_FIT_LIMIT_SECONDS:\n                failure_reason, failure_observed = "fit_seconds", phase_elapsed\n            elif phase == "predict" and phase_elapsed > RF_PREDICT_LIMIT_SECONDS:\n                failure_reason, failure_observed = "predict_seconds", phase_elapsed\n\n            if failure_reason is not None:\n                p6._terminate_process_tree(process)\n                break\n            time.sleep(RF_MONITOR_INTERVAL_SECONDS)\n\n        stdout, stderr = popen.communicate(timeout=30)\n        elapsed = time.monotonic() - started\n        monitoring = pd.DataFrame(\n            [\n                {\n                    "candidate_id": "RF02",\n                    "run_kind": "final_development_refit_and_test_evaluation",\n                    "isolated_worker": True,\n                    "monitor_interval_seconds": RF_MONITOR_INTERVAL_SECONDS,\n                    "peak_process_tree_rss_bytes": peak_rss,\n                    "rss_limit_bytes": rss_limit,\n                    "minimum_available_memory_bytes": minimum_available_bytes,\n                    "minimum_available_memory_percent": minimum_available_percent,\n                    "minimum_allowed_available_bytes": RF_MIN_AVAILABLE_BYTES,\n                    "minimum_allowed_available_percent": RF_MIN_AVAILABLE_PERCENT,\n                    "elapsed_total_seconds": elapsed,\n                    "fit_limit_seconds": RF_FIT_LIMIT_SECONDS,\n                    "predict_limit_seconds": RF_PREDICT_LIMIT_SECONDS,\n                    "total_limit_seconds": RF_TOTAL_LIMIT_SECONDS,\n                    "worker_return_code": popen.returncode,\n                    "resource_failure_reason": failure_reason or "",\n                    "resource_failure_observed": failure_observed\n                    if failure_observed is not None\n                    else np.nan,\n                }\n            ]\n        )\n        if failure_reason is not None:\n            return {\n                "status": "RESOURCE_FAILED",\n                "preflight": pd.DataFrame([preflight]),\n                "monitoring": monitoring,\n                "summary": pd.DataFrame(),\n                "diagnostics": None,\n            }\n        if popen.returncode != 0 or not result_path.exists():\n            raise RuntimeError(\n                "Phase 7 RF02 worker failed outside the approved resource policy. "\n                f"Return code={popen.returncode}; stderr={stderr[-4000:]}"\n            )\n        result = json.loads(result_path.read_text(encoding="utf-8"))\n        diagnostics = {\n            name: pd.DataFrame(rows) for name, rows in result["diagnostics"].items()\n        }\n        summary = pd.DataFrame(\n            [\n                {\n                    **result["configuration"],\n                    "fit_seconds": result["fit_seconds"],\n                    "predict_seconds": result["predict_seconds"],\n                    "total_tree_nodes": result["total_tree_nodes"],\n                    "estimated_tree_bytes": result["estimated_tree_bytes"],\n                    "prediction_shape": str(tuple(result["prediction_shape"])),\n                    "predictions_finite": result["predictions_finite"],\n                    "canonical_checksum_before": result["canonical_checksum_before"],\n                    "canonical_checksum_after": result["canonical_checksum_after"],\n                    "diagnostics_scope": result["diagnostics_scope"],\n                    "status": result["status"],\n                    **diagnostics["overall"].iloc[0].drop(labels=["model"]).to_dict(),\n                }\n            ]\n        )\n        return {\n            "status": "COMPLETE",\n            "preflight": pd.DataFrame([preflight]),\n            "monitoring": monitoring,\n            "summary": summary,\n            "diagnostics": diagnostics,\n        }\n\n', 'phase7_rf_worker.py': '"""Isolated RF02 final-refit worker for Phase 7."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--workspace", required=True)\n    parser.add_argument("--result-path", required=True)\n    parser.add_argument("--status-path", required=True)\n    args = parser.parse_args()\n\n    workspace = Path(args.workspace).resolve()\n    code_dir = workspace / "code"\n    if str(code_dir) not in sys.path:\n        sys.path.insert(0, str(code_dir))\n\n    import phase7_evaluation as evaluation\n\n    evaluation.rf02_worker_run(\n        workspace=workspace,\n        result_path=Path(args.result_path),\n        status_path=Path(args.status_path),\n    )\n\n\nif __name__ == "__main__":\n    main()\n'}
EXPECTED_MODULE_SHA256 = {'feature_engineering.py': '43b94a91e72487d7b8e817f857722d5f36407cdf2e964f00d1188e2cee52cdaf', 'phase6_experiments.py': '1463790d91206116a993b09115e490f65edc695c32b4ac72028130454669db35', 'phase6_rf_worker.py': '7fe8551e1e3fd381ab8549dc17a4fc5930b5477d2e65f518e9bdca29561ba91e', 'phase7_evaluation.py': '915c58e70a33490e62e3db39fe40dc7c93202985f8e929b1e9ab733a97a05c61', 'phase7_rf_worker.py': '2de7ecd201203247dc92947c766876663decbb740ae3a8341f35c4ab324fab2d'}
for module_name, source in MODULE_SOURCES.items():
    path = CODE_DIR / module_name
    path.write_text(source, encoding='utf-8')
    observed = hashlib.sha256(path.read_bytes()).hexdigest()
    if observed != EXPECTED_MODULE_SHA256[module_name]:
        raise AssertionError(f'Module hash mismatch for {module_name}')
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
import feature_engineering as fe
import phase6_experiments as p6
import phase7_evaluation as p7
print(pd.DataFrame({'module': list(EXPECTED_MODULE_SHA256), 'sha256': list(EXPECTED_MODULE_SHA256.values())}).to_string(index=False))


## 3. Fixed-release download and verification

The CSV is streamed to a temporary file. Exact file size and SHA-256 are checked
before the file is promoted and before pandas is called. A mismatch deletes the
temporary download and stops the notebook.


In [ ]:
DATA_URL = "https://github.com/Yai-Git/nsw-electricity-price-data-a2/releases/download/a2-data-v1/prices_NSW1_5min_prepared.csv"
EXPECTED_DATA_SIZE = 14238528
EXPECTED_DATA_SHA256 = "d9a3059fb517eabbdabfa2016b1195fef29eeff0bce00c0b60d7defeab0cddb7"
DATA_PATH = DATA_DIR / "prices_NSW1_5min_prepared.csv"
TEMP_DOWNLOAD = DATA_DIR / "prices_NSW1_5min_prepared.csv.download.tmp"

request = urllib.request.Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
digest = hashlib.sha256()
downloaded = 0
try:
    with urllib.request.urlopen(request, timeout=120) as response, TEMP_DOWNLOAD.open("wb") as handle:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            handle.write(chunk)
            digest.update(chunk)
            downloaded += len(chunk)
    observed_sha = digest.hexdigest()
    if downloaded != EXPECTED_DATA_SIZE or observed_sha != EXPECTED_DATA_SHA256:
        raise RuntimeError(
            "Downloaded dataset failed verification: "
            f"size={downloaded} (expected {EXPECTED_DATA_SIZE}), "
            f"SHA-256={observed_sha} (expected {EXPECTED_DATA_SHA256})."
        )
    TEMP_DOWNLOAD.replace(DATA_PATH)
except Exception:
    TEMP_DOWNLOAD.unlink(missing_ok=True)
    raise

print({"url": DATA_URL, "size_bytes": downloaded, "sha256": observed_sha})


## 4. Dataset, time convention and continuity

Timestamps remain timezone-naive and are interpreted as fixed NEM time/AEST without
daylight-saving conversion. Genuine price spikes and the two real gaps are retained;
window construction is segment-aware so no sample can cross a gap.


In [ ]:
data = fe.load_canonical_dataset(DATA_PATH)
assert fe.sha256_file(DATA_PATH) == EXPECTED_DATA_SHA256
assert DATA_PATH.stat().st_size == EXPECTED_DATA_SIZE
assert data.shape == (348_371, 7)
assert data.columns.tolist() == [
    "date_time", "RRP", "RUNNO", "PERIODID", "PRICE_STATUS", "INVALIDFLAG",
    "continuity_segment_id",
]
assert data["date_time"].iloc[0] == pd.Timestamp("2021-10-01 00:05:00")
assert data["date_time"].iloc[-1] == pd.Timestamp("2025-01-22 15:50:00")
assert data["date_time"].dt.tz is None
assert data["continuity_segment_id"].nunique() == 3
assert data["RRP"].notna().all()

segment_check = data.groupby("continuity_segment_id").agg(
    first_timestamp=("date_time", "min"), last_timestamp=("date_time", "max"),
    records=("date_time", "size"),
).reset_index()
print(segment_check.to_string(index=False))


## 5. Forecasting task and leakage-safe feature builder

For an origin `t`, the raw history is the 288 prices from `t-287` through `t` and the
target is the 96 prices from `t+1` through `t+96`. The 134-column F3 vector contains,
in order, 12 recent lags, six origin-calendar features, 20 rolling statistics and 96
previous-day aligned prices. F1 is columns 0–17 and F2 is columns 0–37. All price
features end at the forecast origin. Each complete 384-record sample must remain in
one continuity segment and one chronological split.


In [ ]:
assert len(fe.FEATURE_NAMES) == len(set(fe.FEATURE_NAMES)) == 134
assert {name: (value.start, value.stop) for name, value in fe.FEATURE_SET_SLICES.items()} == {
    "F1": (0, 18), "F2": (0, 38), "F3": (0, 134)
}
assert fe.feature_catalog()["source_max_offset"].le(0).all()

training = fe.build_split_matrices(data, "training")
validation = fe.build_split_matrices(data, "validation")
assert training.X.shape == (235_575, 134)
assert validation.X.shape == (52_033, 134)
assert training.Y.shape == (235_575, 96)
assert validation.Y.shape == (52_033, 96)
assert np.shares_memory(training.F1, training.F3)
assert np.shares_memory(training.F2, training.F3)
mapping_checks = fe.representative_validation(data, training, validation)
assert mapping_checks["status"].eq("PASS").all()
assert mapping_checks["all_96_previous_day_mappings_checked"].eq(96).all()
assert mapping_checks["all_96_target_mappings_checked"].eq(96).all()
print({"training_windows": len(training.X), "validation_windows": len(validation.X)})


## 6. Validation experiments

The seasonal-naive forecast uses the 96 previous-day F3 columns. Ridge applies a
training-only `StandardScaler(with_mean=True, with_std=True, copy=True)` and the fixed
alpha grid `[1, 10, 100, 1000, 10000, 100000]` to F1, F2 and F3. The six approved
forests run sequentially in isolated monitored workers. Overall validation MAE is the
only selection metric. Full horizon and extreme diagnostics are retained only for the
baseline, selected F3 Ridge and selected forest.


In [ ]:
representatives = mapping_checks
baseline_prediction, baseline_validation = p6.validate_baseline(
    data, training, validation, representatives
)
assert baseline_validation["status"].eq("PASS").all()
baseline_metrics = p6.overall_metrics(baseline_prediction, validation.Y, baseline_prediction)

ridge = p6.run_ridge_experiments(training, validation, baseline_prediction)
ridge_candidates = ridge["candidates"]
ridge_convergence = ridge["convergence"]
ridge_scalers = ridge["scalers"]
ridge_ablation = ridge["selected"]
selected_f3_prediction = ridge.pop("selected_f3_prediction")
selected_f3 = ridge_ablation.query("feature_set == 'F3'").iloc[0]
selected_f3_label = f"Ridge F3 (alpha={int(selected_f3['alpha'])})"
validation_diagnostics = p6.selected_diagnostics(
    {"Seasonal naive": baseline_prediction, selected_f3_label: selected_f3_prediction},
    validation.Y, baseline_prediction, validation.origin_timestamps,
)

assert ridge_candidates.groupby("feature_set").size().to_dict() == {"F1": 6, "F2": 6, "F3": 6}
assert ridge_scalers["status"].eq("PASS").all()
assert ridge_scalers["attributes_unchanged_after_validation_transform"].all()
assert len(ridge_convergence) == 18 * 96
assert not ridge_ablation["any_target_reached_max_iter"].any()
assert ridge_ablation.set_index("feature_set")["selection_status"].to_dict() == {
    "F1": "UPPER_BOUNDARY_SELECTION", "F2": "INTERIOR_SELECTION",
    "F3": "INTERIOR_SELECTION",
}

del selected_f3_prediction, baseline_prediction, representatives, training, validation, data, ridge
gc.collect()

rf_search = p6.run_random_forest_search(RUNTIME_ROOT)
if rf_search["search_status"] == "RESOURCE_FAILED":
    row = rf_search["preflight"].iloc[0]
    raise RuntimeError(
        "Random Forest preflight/resource policy prevented a valid forest. "
        f"Available={{int(row['available_memory_bytes'])}} bytes / "
        f"{{row['available_memory_percent']:.2f}}%; required at least 4 GiB and 30%. "
        "Use an environment with at least 5 GiB available and rerun from the top."
    )
assert rf_search["search_status"] in {"COMPLETE", "RESOURCE_CONSTRAINED"}
assert rf_search["selected"]["candidate_id"] == "RF02"
assert rf_search["diagnostics_status"] == "COMPLETE"
assert rf_search["candidate_diagnostics_calculated"] is False

rf_label = "Random Forest (RF02)"
for name, table in rf_search["selected_diagnostics"].items():
    selected = table.copy()
    selected["model"] = rf_label
    validation_diagnostics[name] = pd.concat([validation_diagnostics[name], selected], ignore_index=True)
print(ridge_ablation[["feature_set", "alpha", "validation_mae", "selection_status"]].to_string(index=False))
print(validation_diagnostics["overall"].to_string(index=False))


## 7. Frozen validation assertions

Price metrics use `np.isclose(..., atol=0.01, rtol=1e-8)`. Skill is stored as a
fraction and uses absolute tolerance `1e-5`. Exact items—counts, timestamps, hashes,
features, configurations, selections and status labels—must match exactly.


In [ ]:
PRICE_ATOL = 0.01
PRICE_RTOL = 1e-8
SKILL_ATOL = 1e-5
RATE_ATOL_PP = 1e-6

def assert_price(label, observed, expected):
    if not np.isclose(observed, expected, atol=PRICE_ATOL, rtol=PRICE_RTOL):
        raise AssertionError(f"{label}: observed={observed}, expected={expected}, atol={PRICE_ATOL}, rtol={PRICE_RTOL}")

def assert_skill(label, observed, expected):
    if not abs(observed - expected) <= SKILL_ATOL:
        raise AssertionError(f"{label}: observed={observed}, expected={expected}, atol={SKILL_ATOL}")

EXPECTED_VALIDATION = {
    "Seasonal naive": (99.0835772534073, 861.0204349309934, -0.2214238661064291, 0.0),
    "Ridge F3 (alpha=1000)": (83.07348985231368, 638.6107509380435, -6.543671674253933, 0.1615816449596652),
    "Random Forest (RF02)": (69.21925198642431, 642.6264651232186, -22.42769076965997, 0.3014054003178009),
}
validation_overall = validation_diagnostics["overall"].set_index("model")
assert set(validation_overall.index) == set(EXPECTED_VALIDATION)
for model, (mae, rmse, bias, skill) in EXPECTED_VALIDATION.items():
    row = validation_overall.loc[model]
    assert_price(f"{model} validation MAE", row["validation_mae"], mae)
    assert_price(f"{model} validation RMSE", row["validation_rmse"], rmse)
    assert_price(f"{model} validation bias", row["validation_bias"], bias)
    assert_skill(f"{model} validation skill", row["validation_mae_skill"], skill)
print("Frozen validation assertions: PASS")


## 8. Frozen development refit and held-out reproduction

Only after RF02 and F3 Ridge alpha 1000 are confirmed are the unified development and
held-out test samples constructed. The development period adds 383 windows across the
former training-validation boundary. Baseline and Ridge aggregates are calculated and
their row-level objects are discarded before one monitored RF02 worker reconstructs
the matrices and returns aggregate diagnostics only. Test results never feed back into
the design.


In [ ]:
data = fe.load_canonical_dataset(DATA_PATH)
development = fe.build_split_matrices(data, "development")
test = fe.build_split_matrices(data, "test")
assert development.X.shape == (287_991, 134)
assert test.X.shape == (58_848, 134)
assert pd.Timestamp(test.origin_timestamps[0]) == pd.Timestamp("2024-07-01 23:55:00")
assert pd.Timestamp(test.origin_timestamps[-1]) == pd.Timestamp("2025-01-22 07:50:00")

test_mapping = p7.mapping_validation(data, development, test)
assert test_mapping["status"].eq("PASS").all()
baseline_prediction, test_baseline_checks = p7.validate_test_baseline(data, test, test_mapping)
assert test_baseline_checks["status"].eq("PASS").all()
ridge_final = p7.fit_final_ridge(development, test, baseline_prediction)
ridge_prediction = ridge_final.pop("prediction")
test_diagnostics = p7.selected_test_diagnostics(
    {"Seasonal naive": baseline_prediction, "Ridge F3 (alpha=1000)": ridge_prediction},
    test.Y, baseline_prediction, test.origin_timestamps,
)
primary_union = test_diagnostics["extreme"].query(
    "model == 'Seasonal naive' and event_group == 'primary_union'"
).iloc[0]
assert int(primary_union["forecast_target_pair_count"]) == 22_492
assert int(primary_union["unique_realised_timestamp_count"]) == 235

del ridge_prediction, baseline_prediction, development, test, data, ridge_final, test_mapping
gc.collect()

rf_final = p7.run_monitored_rf02(RUNTIME_ROOT)
if rf_final["status"] != "COMPLETE":
    raise RuntimeError(
        "Frozen RF02 held-out reproduction did not complete under the approved resource policy. "
        "No threshold will be reduced; rerun in an environment with at least 5 GiB available."
    )
for name, table in rf_final["diagnostics"].items():
    test_diagnostics[name] = pd.concat([test_diagnostics[name], table], ignore_index=True)
test_overall = test_diagnostics["overall"]
print(test_overall.to_string(index=False))


## 9. Frozen held-out assertions and interpretation safeguards

Overall test evidence is descriptive final evidence, not a tuning objective. Extreme
subsets are defined from observed targets only. Forecast-target pair counts are kept
separate from unique realised timestamps; sparse subsets are interpreted cautiously.


In [ ]:
EXPECTED_TEST = {
    "Seasonal naive": (110.7596765016848, 937.4267548797606, 0.2639478880802496, 0.0),
    "Ridge F3 (alpha=1000)": (97.51044861538018, 671.4195353623085, -11.774219531556314, 0.1196214028857611),
    "Random Forest (RF02)": (99.9663959086492, 689.5252389412318, 11.312547909112247, 0.0974477439257546),
}
test_indexed = test_overall.set_index("model")
assert set(test_indexed.index) == set(EXPECTED_TEST)
for model, (mae, rmse, bias, skill) in EXPECTED_TEST.items():
    row = test_indexed.loc[model]
    assert_price(f"{model} test MAE", row["test_mae"], mae)
    assert_price(f"{model} test RMSE", row["test_rmse"], rmse)
    assert_price(f"{model} test bias", row["test_bias"], bias)
    assert_skill(f"{model} test skill", row["test_mae_skill"], skill)

EXPECTED_EVENT_UNIQUE = {
    "high": 828, "extreme_high": 234, "negative": 7380,
    "very_negative": 6, "extreme_low": 1, "primary_union": 235,
}
events = test_diagnostics["extreme"]
observed_unique = events.query("model == 'Seasonal naive'").set_index("event_group")["unique_realised_timestamp_count"].astype(int).to_dict()
assert observed_unique == EXPECTED_EVENT_UNIQUE
assert p6.EXTREME_RULE_LABELS == {
    "high": "Y >= 300", "extreme_high": "Y >= 1000", "negative": "Y < 0",
    "very_negative": "Y <= -100", "extreme_low": "Y <= -1000",
    "primary_union": "Y <= -1000 or Y >= 1000",
}
assert fe.sha256_file(DATA_PATH) == EXPECTED_DATA_SHA256
print("Frozen held-out assertions: PASS")


## 10. Reproducibility manifest and cleanup

The final cell records compact aggregate evidence, confirms resource-policy constants,
checks that no prohibited serialized artifact exists, and deletes remaining row-level
objects. The notebook repository contains only this notebook and its README; runtime
files remain transient.


In [ ]:
assert p6.RF_PREFLIGHT_MIN_AVAILABLE_BYTES == 4 * 1024**3
assert p6.RF_PREFLIGHT_MIN_AVAILABLE_PERCENT == 30.0
assert p6.RF_MIN_AVAILABLE_BYTES == 2 * 1024**3
assert p6.RF_MIN_AVAILABLE_PERCENT == 15.0
assert p6.RF_FIT_LIMIT_SECONDS == 1800.0
assert p6.RF_PREDICT_LIMIT_SECONDS == 300.0
assert p6.RF_TOTAL_LIMIT_SECONDS == 2100.0
assert p6.RF_MONITOR_INTERVAL_SECONDS == 1.0

manifest = {
    "dataset_url": DATA_URL,
    "dataset_size_bytes": EXPECTED_DATA_SIZE,
    "dataset_sha256": fe.sha256_file(DATA_PATH),
    "feature_count": len(fe.FEATURE_NAMES),
    "validation_status": rf_search["search_status"],
    "validation_selected_forest": rf_search["selected"]["candidate_id"],
    "test_selected_ridge_alpha": p7.RIDGE_ALPHA,
    "test_selected_forest": p7.RF02_CONFIG["candidate_id"],
    "test_first_origin": "2024-07-01 23:55:00",
    "test_last_origin": "2025-01-22 07:50:00",
    "frozen_assertions": "PASS",
}

del test_diagnostics, test_overall, test_indexed, events, validation_diagnostics
del rf_search, rf_final, mapping_checks, baseline_validation
gc.collect()

forbidden_suffixes = {".npy", ".npz", ".pkl", ".pickle", ".joblib", ".parquet", ".feather"}
forbidden = [p for p in RUNTIME_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in forbidden_suffixes]
if forbidden:
    raise AssertionError(f"Forbidden persisted artifacts: {forbidden}")
print(json.dumps(manifest, indent=2))


## Frozen result summary

| Model | Validation MAE | Validation skill | Test MAE | Test skill |
|---|---:|---:|---:|---:|
| Seasonal naive | 99.08 | 0.00% | 110.76 | 0.00% |
| Ridge F3, alpha 1000 | 83.07 | 16.16% | **97.51** | **11.96%** |
| Random Forest RF02 | **69.22** | **30.14%** | 99.97 | 9.74% |

RF02 was the validation winner; Ridge was slightly better on the untouched test
period. This difference is reported as temporal generalisation uncertainty, not as
permission to retune. The nested F1–F3 Ridge results are incremental ablation evidence,
not causal feature importance. High-price targets were generally underpredicted, while
both machine-learning models were worse than seasonal naive on negative-price targets.
